Исторический исследовательский ноутбук. Сохранён как запись этапов работы; выводы очищены, код не изменён. Ячейки содержат альтернативные сценарии и старые пути. Для повторения актуальных расчётов используйте `dissertation/run_analysis.py` и инструкции в корневом README.


# Влияния зависимости PARL (представленность женщин в парламенте) на HCI (человеческий капитал)

## Библиотеки

In [ ]:
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

from linearmodels.panel import PanelOLS

import statsmodels.formula.api as smf

import plotly.express as px

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# какая-то штука для красивого отображения циферок
pd.set_option('display.float_format', '{:,.2f}'.format)

## Данные

In [ ]:
df = pd.read_csv('../data/final_data/final_data.csv', sep = ';', decimal = ',')
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# доля пропусков
print((df[['PARL', 'Human_Capital']].isnull().mean() * 100))

In [ ]:
# сколько пропусков в PARL по годам
parl_null_by_year = (
    df
    .groupby('Year')
    .agg(
        n_total=('PARL', 'size'),
        n_null=('PARL', lambda s: s.isna().sum())
    )
    .reset_index()
)

parl_null_by_year['null_share'] = parl_null_by_year['n_null'] / parl_null_by_year['n_total']
parl_null_by_year.sort_values('Year')

In [ ]:
# выкидываю все, что до 1997-го года, так как там нет значений PARL
df = df[df.Year >= 1997]

In [ ]:
# сколько пропусков в PARL по странам
parl_null_by_country = (
    df
    .groupby('Country_Code')
    .agg(
        Country_Name=('Country_Name', 'first'),
        n_years=('PARL', 'size'),
        n_null=('PARL', lambda s: s.isna().sum())
    )
    .reset_index()
)

parl_null_by_country['null_share'] = parl_null_by_country['n_null'] / parl_null_by_country['n_years']
parl_null_by_country.sort_values('null_share', ascending=False)

In [ ]:
# сколько пропусков в Human_Capital по годам
hc_null_by_year = (
    df
    .groupby('Year')
    .agg(
        n_total=('Human_Capital', 'size'),
        n_null=('Human_Capital', lambda s: s.isna().sum())
    )
    .reset_index()
)

hc_null_by_year['null_share'] = hc_null_by_year['n_null'] / hc_null_by_year['n_total']
hc_null_by_year.sort_values('Year')

In [ ]:
# сколько пропусков в Human_Capital по странам
hc_null_by_country = (
    df
    .groupby('Country_Code')
    .agg(
        Country_Name=('Country_Name', 'first'),
        n_years=('Human_Capital', 'size'),
        n_null=('Human_Capital', lambda s: s.isna().sum())
    )
    .reset_index()
)

hc_null_by_country['null_share'] = hc_null_by_country['n_null'] / hc_null_by_country['n_years']
hc_null_by_country.sort_values('null_share', ascending=False)

In [ ]:
# страны, у которых Human_Capital = NaN
bad_countries = hc_null_by_country.loc[hc_null_by_country['null_share'] == 1, 'Country_Name']

# что это за страны?
bad_countries_list = bad_countries.tolist()
print(len(bad_countries_list), 'стран будут исключены:')
print(bad_countries_list)

# фильтрую датасет
df_analysis = df[~df['Country_Name'].isin(bad_countries_list)].copy()

In [ ]:
# доля пропусков
print((df_analysis[['PARL', 'Human_Capital']].isnull().mean() * 100))

In [ ]:
# колонки с данными для анализа
features = ['Human_Capital', 'PARL', 'GDP', 'Political_Stability', 'Rule_of_Law', 'Fertility', 'Corruption']

# матрица корелляций
corr_matrix = df_analysis[features].corr()

In [ ]:
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu',
    title='Корреляции между числовыми переменными',
    template='plotly_dark'
)

fig.update_layout(
    height=600,
)

fig.show()

## Кластеризация

In [ ]:
# последние 5 лет PARL
max_year = df_analysis['Year'].max()
start_5 = max_year - 4
mask_5 = df_analysis['Year'].between(start_5, max_year)

# фильтрую датасет
df_recent_5 = df_analysis[mask_5].copy()

print('5-летнее окно (Human_Capital / PARL):')
print(df_recent_5[['PARL', 'Human_Capital']].isnull().mean() * 100)

In [ ]:
country_parl_5 = (
    df_recent_5
    .groupby(['Country_Code', 'Country_Name'], as_index=False)
    .agg(
        parl_mean_5=('PARL', 'mean'),
        parl_n_years_5=('PARL', lambda s: s.notna().sum())
    )
)

In [ ]:
country_parl_5['parl_mean_5_q3'] = pd.qcut(
    country_parl_5['parl_mean_5'],
    q=3,
    labels=['low', 'mid', 'high']
)

In [ ]:
df_analysis = df_analysis.merge(
    country_parl_5[['Country_Code', 'parl_mean_5', 'parl_mean_5_q3']],
    on='Country_Code',
    how='left'
)

In [ ]:
X = country_parl_5[['parl_mean_5']].values  # или .to_numpy()

results = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertia = km.inertia_
    sil = silhouette_score(X, labels)
    results.append({'k': k, 'inertia': inertia, 'silhouette': sil})

results_df = pd.DataFrame(results)
results_df

In [ ]:
X = country_parl_5[['parl_mean_5']].values

results_gmm = []
for k in range(1, 7):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(X)
    bic = gmm.bic(X)
    aic = gmm.aic(X)
    results_gmm.append({'k': k, 'bic': bic, 'aic': aic})

results_gmm_df = pd.DataFrame(results_gmm)
results_gmm_df

In [ ]:
# создаю бины на основании имеющихся исследований для кластеризации стран
bins = [-float('inf'), 15, 30, 40, float('inf')]
labels = ['low', 'medium', 'high', 'very_high']

# кластеризую
country_parl_5['parl_level_4'] = pd.cut(
    country_parl_5['parl_mean_5'],
    bins=bins,
    labels=labels
)

In [ ]:
df_analysis = df_analysis.merge(
    country_parl_5[['Country_Code', 'parl_level_4']],
    on='Country_Code',
    how='left'
)

In [ ]:
# одна строка на страну (если есть Country_Name – тоже заберём)
cols = ['Country_Code', 'parl_level_4']
if 'Country_Name' in df_analysis.columns:
    cols.insert(1, 'Country_Name')

country_clusters = (
    df_analysis[cols]
    .drop_duplicates()
    .sort_values(['parl_level_4', 'Country_Code'])
)

for cl, sub in country_clusters.groupby('parl_level_4'):
    print(f'\n=== Кластер: {cl} ===')
    if 'Country_Name' in sub.columns:
        for _, row in sub.iterrows():
            print(f"{row['Country_Code']} — {row['Country_Name']}")
    else:
        print(', '.join(sorted(sub['Country_Code'].unique())))

In [ ]:
fig = px.box(
    country_parl_5,
    x='parl_level_4',
    y='parl_mean_5',
    points='all',
    title='Средний PARL за 2018–2022 по квантильным группам',
    template='plotly_dark',
    category_orders={'parl_level_4': ['low', 'medium', 'high', 'very_high']}
)

fig.update_layout(
    xaxis_title='Квантильная группа',
    yaxis_title='Средний PARL за 5 лет',
    height=600
)

fig.show()

In [ ]:
n_countries_total = df_recent_5['Country_Code'].nunique()
print('Всего стран в окне 2018–2022:', n_countries_total)

In [ ]:
# сколько стран есть в country_parl_5 и сколько из них с ненулевым кластером
n_countries_cluster_table = country_parl_5['Country_Code'].nunique()
n_countries_with_cluster = country_parl_5[country_parl_5['parl_level_4'].notna()]['Country_Code'].nunique()

print('Стран в таблице country_parl_5:', n_countries_cluster_table)
print('Стран, у которых задан parl_level_4:', n_countries_with_cluster)

In [ ]:
# распределение по кластерам
print(country_parl_5['parl_level_4'].value_counts(dropna=False))

# явно посмотреть тех, у кого кластера нет
missing_cluster = country_parl_5[country_parl_5['parl_level_4'].isna()]

In [ ]:
print('Стран без кластера:', missing_cluster['Country_Code'].nunique())
missing_cluster[['Country_Code', 'Country_Name', 'parl_mean_5']].head()

## Модели

In [ ]:
features = ['Human_Capital', 'PARL', 'GDP', 'Political_Stability', 'Fertility', 'Corruption']

In [ ]:
# рабочий датасет для моделей
df_model = df_analysis[['Country_Code', 'Year'] + features].copy()

In [ ]:
# выкидываем NaNы
model_vars = ['Country_Code', 'Year'] + features
df_fe = df_model[model_vars].dropna().copy()

print('Размер исходного df_model:', df_model.shape)
print('Размер очищенного df_fe:', df_fe.shape)
print('Число стран в df_fe:', df_fe["Country_Code"].nunique())

In [ ]:
formula_fe = (
    'Human_Capital ~ PARL + GDP + Political_Stability + '
    'Fertility + Corruption + '
    'C(Country_Code) + C(Year)'
)

fe_model = smf.ols(formula=formula_fe, data=df_fe).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_fe['Country_Code']}
)

print(fe_model.summary())
print('Число наблюдений в модели:', fe_model.nobs)

In [ ]:
# Берём нужные колонки
df_panel = df_analysis[['Country_Code', 'Year', 'Human_Capital', 'parl_level_4'] + features].copy()

# PanelOLS всё равно выкинет строки с NaN → делаем это явно
df_panel = df_panel.dropna(subset=['Human_Capital'] + features)

# Делаем панельный MultiIndex (страна, год)
df_panel = df_panel.set_index(['Country_Code', 'Year']).sort_index()

In [ ]:
print(df_panel['parl_level_4'].value_counts())
print('\nЧисло стран по кластерам:')
print(df_panel.reset_index().groupby('parl_level_4')['Country_Code'].nunique())

In [ ]:
def run_panel_ols_for_cluster(cluster_label):
    # 1. фильтруем по кластеру
    sub = df_panel[df_panel['parl_level_4'] == cluster_label].copy()

    # для спокойствия оставим только нужные колонки
    cols = ['Human_Capital'] + features
    sub = sub[cols].dropna()

    n_countries = sub.index.get_level_values('Country_Code').nunique()
    n_obs = sub.shape[0]
    print(f'\n=== Кластер: {cluster_label} ===')
    print(f'Стран: {n_countries}, наблюдений: {n_obs}')

    if n_countries < 5:
        print('⚠️ Мало стран в кластере, результаты будут шумными')

    # 2. все числовое
    sub = sub.astype(float)

    # 3. PanelOLS по формуле
    # EntityEffects = фиксированные эффекты по странам
    # TimeEffects НЕ добавляем (как и хотели)
    mod = PanelOLS.from_formula(
        'Human_Capital ~ PARL + GDP + Political_Stability + '
        'Fertility + Corruption + EntityEffects',
        data=sub
    )

    res = mod.fit(
        cov_type='clustered',
        cluster_entity=True
    )

    print(res.summary)
    return res

In [ ]:
results_by_cluster = {}

for cl in ['low', 'medium', 'high', 'very_high']:
    results_by_cluster[cl] = run_panel_ols_for_cluster(cl)

In [ ]:
df_panel_all = df_analysis[['Country_Code', 'Year', 'Human_Capital'] + features].copy()
df_panel_all = df_panel_all.dropna(subset=['Human_Capital'] + features)

# Делаем панельный MultiIndex (страна, год)
df_panel_all = df_panel_all.set_index(['Country_Code', 'Year']).sort_index()

# На всякий случай всё в float
df_panel_all = df_panel_all.astype(float)

In [ ]:
mod_all = PanelOLS.from_formula(
    'Human_Capital ~ PARL + GDP + Political_Stability + '
    'Fertility + Corruption + EntityEffects',
    data=df_panel_all
)

res_all = mod_all.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(res_all.summary)

## Причинность

In [ ]:
# скопируем нужные колонки (если хочешь, можно взять сразу весь df_analysis)
df_lag = df_analysis.copy()

# выбираем длину лага
L_15 = 15
L_10 = 10  # 10 лет лага, как первый вариант
L_5 = 5

# создаём лаг PARL внутри каждой страны
df_lag = df_lag.sort_values(['Country_Code', 'Year'])

df_lag['PARL_lag15'] = (
    df_lag
    .groupby('Country_Code')['PARL']
    .shift(L_15)   # сдвиг на 15 лет назад
)

df_lag['PARL_lag10'] = (
    df_lag
    .groupby('Country_Code')['PARL']
    .shift(L_10)   # сдвиг на 10 лет назад
)

df_lag['PARL_lag5'] = (
    df_lag
    .groupby('Country_Code')['PARL']
    .shift(L_5)   # сдвиг на 5 лет назад
)

# Делаем MultiIndex (Country_Code, Year) как требует PanelOLS
df_lag = df_lag.set_index(['Country_Code', 'Year']).sort_index()

In [ ]:
# Убираем строки, где есть NaN хотя бы в одной из переменных модели
df_lag10 = df_lag.dropna(subset=[
    'Human_Capital',
    'PARL_lag10',
    'GDP',
    'Political_Stability',
    'Fertility',
    'Corruption'
])

# FE-модель с лагом PARL_lag10
mod_lag_10 = PanelOLS.from_formula(
    'Human_Capital ~ PARL_lag10 + GDP + Political_Stability + '
    'Fertility + Corruption + EntityEffects',
    data=df_lag10
)

res_lag_10 = mod_lag_10.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(res_lag_10.summary)

In [ ]:
# Убираем строки, где есть NaN хотя бы в одной из переменных модели
df_lag5 = df_lag.dropna(subset=[
    'Human_Capital',
    'PARL_lag5',
    'GDP',
    'Political_Stability',
    'Fertility',
    'Corruption'
])

# FE-модель с лагом PARL_lag5
mod_lag_5 = PanelOLS.from_formula(
    'Human_Capital ~ PARL_lag5 + GDP + Political_Stability + '
    'Fertility + Corruption + EntityEffects',
    data=df_lag5
)

res_lag_5 = mod_lag_5.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(res_lag_5.summary)

In [ ]:
# Убираем строки, где есть NaN хотя бы в одной из переменных модели
df_lag15 = df_lag.dropna(subset=[
    'Human_Capital',
    'PARL_lag15',
    'GDP',
    'Political_Stability',
    'Fertility',
    'Corruption'
])

# FE-модель с лагом PARL_lag15
mod_lag_15 = PanelOLS.from_formula(
    'Human_Capital ~ PARL_lag15 + GDP + Political_Stability + '
    'Fertility + Corruption + EntityEffects',
    data=df_lag15
)

res_lag_15 = mod_lag_15.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(res_lag_15.summary)

In [ ]:
mod_dyn_5 = PanelOLS.from_formula(
    'Human_Capital ~ PARL + PARL_lag5 + GDP + Political_Stability + '
    'Fertility + Corruption + EntityEffects',
    data=df_lag5
)

res_dyn_5 = mod_dyn_5.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(res_dyn_5.summary)

## Робастность

In [ ]:
mod_lag_5_tw = PanelOLS.from_formula(
    'Human_Capital ~ PARL_lag5 + GDP + Political_Stability + '
    'Fertility + Corruption + EntityEffects + TimeEffects',
    data=df_lag5
)

res_lag_5_tw = mod_lag_5_tw.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(res_lag_5_tw.summary)

In [ ]:
clusters = sorted(df_lag5['parl_level_4'].unique())
results_by_cluster = {}

for cl in clusters:
    df_cl = df_lag5[df_lag5['parl_level_4'] == cl]

    n_countries = df_cl.index.get_level_values(0).nunique()
    n_obs = df_cl.shape[0]
    print(f"\n=== Кластер: {cl} ===")
    print(f"Стран: {n_countries}, наблюдений: {n_obs}")

    mod_cl = PanelOLS.from_formula(
        'Human_Capital ~ PARL_lag5 + GDP + Political_Stability + '
        'Fertility + Corruption + EntityEffects',
        data=df_cl
    )

    res_cl = mod_cl.fit(
        cov_type='clustered',
        cluster_entity=True
    )

    results_by_cluster[cl] = res_cl
    print(res_cl.summary)

In [ ]:
from pydynpd import regression

df_gmm = df_analysis.rename(columns={
    "Country_Code": "id",
    "Year": "year",
    "Human_Capital": "hc",
}).copy()

cols = ["id", "year", "hc", "PARL", "Fertility", "Corruption", "GDP", "Political_Stability"]
df_gmm = df_gmm[cols].dropna().sort_values(["id", "year"])

# опционально: оставить страны с достаточной длиной ряда
min_T = 8
df_gmm = df_gmm.groupby("id", group_keys=False).filter(lambda g: len(g) >= min_T)

cmd = (
    "hc L1.hc PARL Fertility Corruption GDP Political_Stability | "
    "gmm(hc, 2:4) gmm(PARL, 2:4) iv(Fertility Corruption GDP Political_Stability) | "
    "timedumm"
)
res = regression.abond(cmd, df_gmm, ["id", "year"])
print(res.models[0].regression_table)

In [ ]:
from pydynpd import regression

df_gmm = df_analysis.rename(columns={
    "Country_Code": "id",
    "Year": "year",
    "Human_Capital": "hc",
}).copy()

cols = ["id", "year", "hc", "PARL", "Fertility", "Corruption", "GDP", "Political_Stability"]
df_gmm = df_gmm[cols].dropna().sort_values(["id", "year"])

# опционально: оставить страны с достаточной длиной ряда
min_T = 8
df_gmm = df_gmm.groupby("id", group_keys=False).filter(lambda g: len(g) >= min_T)

cmd = (
  "hc L1.hc PARL Fertility Corruption GDP Political_Stability | "
  "gmm(hc, 2:3) gmm(PARL, 2:3) iv(Fertility Corruption GDP Political_Stability) | "
  "timedumm collapse"
)

res = regression.abond(cmd, df_gmm, ["id", "year"])
print(res.models[0].regression_table)

In [ ]:
from pydynpd import regression

df_gmm = df_analysis.rename(columns={
    "Country_Code": "id",
    "Year": "year",
    "Human_Capital": "hc",
}).copy()

cols = ["id", "year", "hc", "PARL", "Fertility", "Corruption", "GDP", "Political_Stability"]
df_gmm = df_gmm[cols].dropna().sort_values(["id", "year"])

# опционально: оставить страны с достаточной длиной ряда
min_T = 8
df_gmm = df_gmm.groupby("id", group_keys=False).filter(lambda g: len(g) >= min_T)

cmd = (
  "hc L1.hc PARL Fertility Corruption GDP Political_Stability | "
  "gmm(hc, 2:3) gmm(PARL, 2:3) iv(Fertility Corruption GDP Political_Stability) | "
  "timedumm collapse onestep"
)

res = regression.abond(cmd, df_gmm, ["id", "year"])
print(res.models[0].regression_table)

In [ ]:
from pydynpd import regression

df_gmm = df_analysis.rename(columns={
    "Country_Code": "id",
    "Year": "year",
    "Human_Capital": "hc",
}).copy()

cols = ["id", "year", "hc", "PARL", "Fertility", "Corruption", "GDP", "Political_Stability"]
df_gmm = df_gmm[cols].dropna().sort_values(["id", "year"])

# опционально: оставить страны с достаточной длиной ряда
min_T = 8
df_gmm = df_gmm.groupby("id", group_keys=False).filter(lambda g: len(g) >= min_T)

cmd = (
  "hc L1.hc PARL Fertility Corruption GDP Political_Stability | "
  "gmm(hc, 2:3) gmm(PARL, 2:3) iv(Fertility Corruption GDP Political_Stability) | "
  "timedumm collapse nolevel"
)

res = regression.abond(cmd, df_gmm, ["id", "year"])
print(res.models[0].regression_table)

In [ ]:
from pydynpd import regression

df_gmm = df_analysis.rename(columns={
    "Country_Code": "id",
    "Year": "year",
    "Human_Capital": "hc",
}).copy()

cols = ["id", "year", "hc", "PARL", "Fertility", "Corruption", "GDP", "Political_Stability"]
df_gmm = df_gmm[cols].dropna().sort_values(["id", "year"])

# опционально: оставить страны с достаточной длиной ряда
min_T = 6
df_gmm = df_gmm.groupby("id", group_keys=False).filter(lambda g: len(g) >= min_T)

cmd = (
  "hc L1.hc PARL Fertility Corruption GDP Political_Stability | "
  "gmm(hc, 2:4) gmm(PARL, 2:4) iv(Fertility Corruption GDP Political_Stability) | "
  "timedumm collapse"
)

res = regression.abond(cmd, df_gmm, ["id", "year"])
print(res.models[0].regression_table)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# ======================
# SETTINGS (можно не менять)
# ======================
DATA_PATH = "../data/final_data/final_data.csv"
YEAR_MIN = 1997

MAIN_OUTCOME = "Life_Expectancy"
SECOND_OUTCOME = "Female_Adult_Literacy"   # если окажется мало данных, позже заменим
ROBUST_OUTCOME = "Human_Capital"

CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]
LAG_YEARS = 5  # long differences

# ======================
# LOAD & CLEAN
# ======================
df = pd.read_csv(DATA_PATH, sep=";", decimal=",")

# numeric cast
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
for c in df.columns:
    if c not in ["Country_Name", "Country_Code", "Region", "Religion"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# filters: Year>=1997
df = df[df["Year"] >= YEAR_MIN].copy()

# drop countries with all Human_Capital missing (как у тебя)
hc_any = df.groupby("Country_Code")["Human_Capital"].apply(lambda s: s.notna().any())
df = df[df["Country_Code"].isin(hc_any[hc_any].index)].copy()

# ======================
# HELPERS
# ======================
def fit_models_for_outcome(data: pd.DataFrame, outcome: str):
    """Fits 3 comparable models on the same sample: Country FE, TWFE, Country FE + trend."""
    cols = ["Country_Code", "Year", "PARL", outcome] + CONTROLS
    d = data[cols].dropna().copy()

    info = {
        "outcome": outcome,
        "n_obs": len(d),
        "n_countries": d["Country_Code"].nunique(),
        "year_min": int(d["Year"].min()) if len(d) else np.nan,
        "year_max": int(d["Year"].max()) if len(d) else np.nan,
    }

    if len(d) == 0:
        return info, pd.DataFrame()

    controls_str = " + ".join(CONTROLS)

    # 1) Country FE
    f1 = f"{outcome} ~ PARL + {controls_str} + C(Country_Code)"
    m1 = smf.ols(f1, data=d).fit(cov_type="cluster", cov_kwds={"groups": d["Country_Code"]})

    # 2) TWFE
    f2 = f"{outcome} ~ PARL + {controls_str} + C(Country_Code) + C(Year)"
    m2 = smf.ols(f2, data=d).fit(cov_type="cluster", cov_kwds={"groups": d["Country_Code"]})

    # 3) Country FE + linear time trend
    f3 = f"{outcome} ~ PARL + {controls_str} + C(Country_Code) + Year"
    m3 = smf.ols(f3, data=d).fit(cov_type="cluster", cov_kwds={"groups": d["Country_Code"]})

    def row(name, model):
        return {
            "model": name,
            "coef_PARL": model.params.get("PARL", np.nan),
            "se": model.bse.get("PARL", np.nan),
            "pval": model.pvalues.get("PARL", np.nan),
            "n_obs": int(model.nobs),
        }

    out = pd.DataFrame([
        row("Country FE", m1),
        row("TWFE (Country+Year FE)", m2),
        row("Country FE + linear trend", m3),
    ])

    return info, out

def fit_long_differences(data: pd.DataFrame, outcome: str, lag_years: int):
    """Long differences: ΔY_{t} = Y_t - Y_{t-lag}; regress ΔY on ΔPARL and Δcontrols."""
    cols = ["Country_Code", "Year", "PARL", outcome] + CONTROLS
    d = data[cols].dropna().sort_values(["Country_Code", "Year"]).copy()

    if len(d) == 0:
        return pd.DataFrame()

    for col in ["PARL", outcome] + CONTROLS:
        d[f"d_{col}"] = d.groupby("Country_Code")[col].diff(lag_years)

    need = [f"d_{outcome}", "d_PARL"] + [f"d_{c}" for c in CONTROLS]
    dd = d.dropna(subset=need).copy()

    if len(dd) == 0:
        return pd.DataFrame()

    rhs = " + ".join([f"d_{c}" for c in CONTROLS])
    f = f"d_{outcome} ~ d_PARL + {rhs}"
    m = smf.ols(f, data=dd).fit(cov_type="cluster", cov_kwds={"groups": dd["Country_Code"]})

    out = pd.DataFrame([{
        "model": f"Long differences ({lag_years}y)",
        "coef_dPARL": m.params.get("d_PARL", np.nan),
        "se": m.bse.get("d_PARL", np.nan),
        "pval": m.pvalues.get("d_PARL", np.nan),
        "n_obs": int(m.nobs),
        "n_countries": int(dd["Country_Code"].nunique()),
        "year_min": int(dd["Year"].min()),
        "year_max": int(dd["Year"].max()),
    }])

    return out

# ======================
# RUN
# ======================
outcomes = [MAIN_OUTCOME, SECOND_OUTCOME, ROBUST_OUTCOME]

print("=== DATA CHECK ===")
print("Years:", int(df["Year"].min()), "-", int(df["Year"].max()))
print("Countries:", df["Country_Code"].nunique())
print()

all_tables = {}

for y in outcomes:
    print(f"=== OUTCOME: {y} ===")
    info, table = fit_models_for_outcome(df, y)
    print("Sample:", info)
    if table.empty:
        print("No data after dropna for this outcome.\n")
        continue
    print(table.to_string(index=False))
    print()
    all_tables[y] = table

print("=== LONG DIFFERENCES for MAIN OUTCOME ===")
ld = fit_long_differences(df, MAIN_OUTCOME, LAG_YEARS)
if ld.empty:
    print("No data for long differences (after diff+dropna).")
else:
    print(ld.to_string(index=False))


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

DATA_PATH = "../data/final_data/final_data.csv"
YEAR_MIN = 1997

OUTCOME = "Life_Expectancy"
CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]

LAGS = [0, 1, 3, 5]      # 0 = contemporaneous PARL
LEADS = [1, 3]          # placebo tests

# ---- load ----
df = pd.read_csv(DATA_PATH, sep=";", decimal=",")

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
for c in df.columns:
    if c not in ["Country_Name", "Country_Code", "Region", "Religion"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[df["Year"] >= YEAR_MIN].copy()

# keep same country filter as before: drop countries with all Human_Capital missing
hc_any = df.groupby("Country_Code")["Human_Capital"].apply(lambda s: s.notna().any())
df = df[df["Country_Code"].isin(hc_any[hc_any].index)].copy()

# basic checks
if "Region" not in df.columns:
    raise ValueError("No column 'Region' found in data.")
if df["Region"].isna().mean() > 0.2:
    print("WARNING: Region has many missing values. Share this info if results look odd.")

# ---- build lags/leads of PARL ----
df = df.sort_values(["Country_Code", "Year"]).copy()
for L in [1, 3, 5]:
    df[f"PARL_lag{L}"] = df.groupby("Country_Code")["PARL"].shift(L)
for K in [1, 3]:
    df[f"PARL_lead{K}"] = df.groupby("Country_Code")["PARL"].shift(-K)

# ---- model runner ----
def run_model(data: pd.DataFrame, parl_var: str):
    # restrict to needed columns and dropna to keep sample explicit
    cols = ["Country_Code", "Year", "Region", OUTCOME, parl_var] + CONTROLS
    d = data[cols].dropna().copy()

    if len(d) == 0:
        return None

    controls_str = " + ".join(CONTROLS)

    # Region×Year FE implemented as interaction fixed effects
    # Country FE included as usual
    formula = f"{OUTCOME} ~ {parl_var} + {controls_str} + C(Country_Code) + C(Region):C(Year)"

    m = smf.ols(formula, data=d).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["Country_Code"]}
    )

    return {
        "parl_var": parl_var,
        "coef": m.params.get(parl_var, np.nan),
        "se": m.bse.get(parl_var, np.nan),
        "pval": m.pvalues.get(parl_var, np.nan),
        "n_obs": int(m.nobs),
        "n_countries": int(d["Country_Code"].nunique()),
        "year_min": int(d["Year"].min()),
        "year_max": int(d["Year"].max()),
    }

rows = []

# contemporaneous and lags
rows.append(run_model(df, "PARL"))
for L in [1, 3, 5]:
    rows.append(run_model(df, f"PARL_lag{L}"))

# placebo leads
for K in [1, 3]:
    rows.append(run_model(df, f"PARL_lead{K}"))

out = pd.DataFrame([r for r in rows if r is not None])
out = out.sort_values(
    by="parl_var",
    key=lambda s: s.map(lambda x: (
        0 if x=="PARL" else
        1 if "lag" in x else
        2
    ))
)

print("=== Region×Year FE + Country FE results ===")
print(out.to_string(index=False))


In [ ]:
# -*- coding: utf-8 -*-
"""
Event study / DiD: quota implementation shock (QAROT) -> PARL (first stage) -> outcomes

Inputs:
- Your panel: ../data/final_data/final_data.csv   (semicolon-separated; decimal comma)
- QAROT (country-year): local file in ./qarot_data/

Outputs (created next to script):
- merged_panel_qarot_1997_2015.csv
- eventstudy_results_PARL.csv
- eventstudy_results_Human_Capital.csv
- eventstudy_results_Life_Expectancy.csv
"""

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -------------------------
# CONFIG
# -------------------------
DATA_PATH = "../data/final_data/final_data.csv"

QAROT_DIR = "../data/qarot_data"
QAROT_FILE = None  # set exact filename if you want, else first .csv in folder

YEAR_MIN = 1997
YEAR_MAX = 2015

K_PRE = 5
K_POST = 10

CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]
OUTCOMES = ["PARL", "Human_Capital", "Life_Expectancy"]

# -------------------------
# HELPERS
# -------------------------
def find_qarot_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path

    if not os.path.isdir(folder):
        raise FileNotFoundError(f"QAROT folder not found: {folder}")

    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")

    return os.path.join(folder, csvs[0])


def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]

    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")

    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year"})
    )

    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year_fallback"})
    )

    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])

    return out


def k_to_col(prefix: str, k: int) -> str:
    """Safe column name for patsy: no '-' sign."""
    if k < 0:
        return f"{prefix}_m{abs(k)}"   # minus
    if k > 0:
        return f"{prefix}_p{k}"        # plus
    return f"{prefix}_0"


def make_event_dummies(df: pd.DataFrame, treat_col="treat_year",
                       k_pre=5, k_post=10, base=-1, prefix="evt"):
    d = df.copy()
    d["ever_treated"] = d[treat_col].notna().astype(int)
    d["rel_year"] = d["Year"] - d[treat_col]

    # Keep treated only within window; keep all never-treated
    in_window = d["rel_year"].between(-k_pre, k_post)
    d = d[(d["ever_treated"] == 0) | in_window].copy()

    ks = list(range(-k_pre, k_post + 1))
    ks = [k for k in ks if k != base]

    for k in ks:
        col = k_to_col(prefix, k)
        d[col] = ((d["ever_treated"] == 1) & (d["rel_year"] == k)).astype(int)

    return d, ks


def fit_event_study(df: pd.DataFrame, y: str, ks: list, controls: list, prefix="evt"):
    event_cols = [k_to_col(prefix, k) for k in ks]
    rhs = event_cols + controls + ["C(Country_Code)", "C(Year)"]
    formula = f"{y} ~ " + " + ".join(rhs)

    needed = [y] + controls + ["Country_Code", "Year"] + event_cols
    tmp = df.dropna(subset=needed).copy()

    model = smf.ols(formula=formula, data=tmp).fit(
        cov_type="cluster",
        cov_kwds={"groups": tmp["Country_Code"]}
    )
    return model, tmp, event_cols


def extract_event_table(model, ks, prefix="evt"):
    rows = []
    for k in ks:
        name = k_to_col(prefix, k)
        if name not in model.params.index:
            continue
        rows.append({
            "k": k,
            "term": name,
            "coef": float(model.params[name]),
            "se": float(model.bse[name]),
            "pval": float(model.pvalues[name]),
        })
    return pd.DataFrame(rows).sort_values("k")


# -------------------------
# MAIN
# -------------------------
def main():
    # Load your panel
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    # Load QAROT
    qarot_path = find_qarot_csv(QAROT_DIR, QAROT_FILE)
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    # Build treat year and merge
    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    print("Panel after merge:", df.shape, "countries:", df["Country_Code"].nunique())
    print("Ever treated (by 2015):", int(df["treat_year"].notna().sum() > 0))

    # Event dummies
    df_es, ks = make_event_dummies(df, treat_col="treat_year", k_pre=K_PRE, k_post=K_POST, base=-1, prefix="evt")

    # Save merged panel
    merged_out = "merged_panel_qarot_1997_2015.csv"
    df_es.to_csv(merged_out, index=False)
    print("Saved:", merged_out)

    # Run models
    for y in OUTCOMES:
        if y not in df_es.columns:
            print(f"Skip {y}: not in dataset columns")
            continue

        model, used, event_cols = fit_event_study(df_es, y=y, ks=ks, controls=CONTROLS, prefix="evt")
        tab = extract_event_table(model, ks, prefix="evt")

        print("\n==============================")
        print("EVENT STUDY:", y)
        print("N obs:", used.shape[0], "countries:", used["Country_Code"].nunique(),
              "years:", used["Year"].min(), "-", used["Year"].max())
        print(tab.to_string(index=False))

        out_csv = f"eventstudy_results_{y}.csv"
        tab.to_csv(out_csv, index=False)
        print("Saved:", out_csv)

        # quick summaries
        pre = tab[tab["k"].between(-K_PRE, -2)]
        post = tab[tab["k"].between(0, min(5, K_POST))]
        if len(pre) and len(post):
            print("Avg lead (-5..-2):", float(pre["coef"].mean()))
            print("Avg post (0..5):", float(post["coef"].mean()))

    print("\nDONE.")


if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -------------------------
# CONFIG
# -------------------------
DATA_PATH = "../data/final_data/final_data.csv"
QAROT_DIR = "../data/qarot_data"
QAROT_FILE = None  # set exact filename if you want

YEAR_MIN = 1997
YEAR_MAX = 2015

K_PRE = 5
K_POST = 10
BASE = -1  # omitted period

CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]
OUTCOMES = ["PARL", "Human_Capital", "Life_Expectancy"]

# -------------------------
# HELPERS
# -------------------------
def find_qarot_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"QAROT folder not found: {folder}")
    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")
    return os.path.join(folder, csvs[0])

def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]
    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")

    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year"})
    )

    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year_fallback"})
    )

    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])
    return out

def k_to_col(prefix: str, k: int) -> str:
    if k < 0:
        return f"{prefix}_m{abs(k)}"
    if k > 0:
        return f"{prefix}_p{k}"
    return f"{prefix}_0"

def make_event_dummies(df: pd.DataFrame, treat_col="treat_year",
                       k_pre=5, k_post=10, base=-1, prefix="evt"):
    d = df.copy()
    d["ever_treated"] = d[treat_col].notna().astype(int)
    d["rel_year"] = d["Year"] - d[treat_col]

    in_window = d["rel_year"].between(-k_pre, k_post)
    d = d[(d["ever_treated"] == 0) | in_window].copy()

    ks = list(range(-k_pre, k_post + 1))
    ks = [k for k in ks if k != base]

    for k in ks:
        col = k_to_col(prefix, k)
        d[col] = ((d["ever_treated"] == 1) & (d["rel_year"] == k)).astype(int)

    return d, ks

def add_bins(df: pd.DataFrame, prefix="evt"):
    """
    Bins for treated:
      pre_bin:   k in [-5,-2]
      post_0_2:  k in [0,2]
      post_3_5:  k in [3,5]
      post_6_10: k in [6,10]
    baseline is -1 (omitted).
    """
    d = df.copy()

    def treated_k_in(a, b):
        return ((d["ever_treated"] == 1) & (d["rel_year"].between(a, b))).astype(int)

    d["bin_pre_m5_m2"] = treated_k_in(-5, -2)
    d["bin_post_0_2"] = treated_k_in(0, 2)
    d["bin_post_3_5"] = treated_k_in(3, 5)
    d["bin_post_6_10"] = treated_k_in(6, 10)

    return d

def fit_ols(df: pd.DataFrame, y: str, rhs_terms: list):
    formula = f"{y} ~ " + " + ".join(rhs_terms)
    needed = [y, "Country_Code", "Year"] + [t for t in rhs_terms if not t.startswith("C(")]
    tmp = df.dropna(subset=list(set(needed))).copy()
    m = smf.ols(formula=formula, data=tmp).fit(
        cov_type="cluster",
        cov_kwds={"groups": tmp["Country_Code"]}
    )
    return m, tmp

def joint_pretrend_test(model, pre_terms: list):
    """
    Returns F-test result for H0: all pre_terms coefficients = 0
    """
    present = [t for t in pre_terms if t in model.params.index]
    if not present:
        return None
    hypothesis = " = 0, ".join(present) + " = 0"
    return model.f_test(hypothesis)

# -------------------------
# MAIN
# -------------------------
def main():
    # Load panel
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    # Load QAROT
    qarot_path = find_qarot_csv(QAROT_DIR, QAROT_FILE)
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    # Treat year
    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    # Diagnostics: treated countries count + treat_year range
    treated_countries = df.loc[df["treat_year"].notna(), "Country_Code"].nunique()
    print("Countries in panel:", df["Country_Code"].nunique())
    print("Treated countries (treat_year notna):", treated_countries)
    if treated_countries > 0:
        print("Treat year min/max:", int(df["treat_year"].min()), "-", int(df["treat_year"].max()))

    # Event dummies (point-by-point)
    df_es, ks = make_event_dummies(df, treat_col="treat_year", k_pre=K_PRE, k_post=K_POST, base=BASE, prefix="evt")

    # Add bins
    df_es = add_bins(df_es)

    # Save merged
    df_es.to_csv("merged_panel_qarot_1997_2015.csv", index=False)

    # ----- Run for each outcome -----
    for y in OUTCOMES:
        if y not in df_es.columns:
            print(f"\nSkip {y}: not in dataset")
            continue

        print("\n==============================")
        print("OUTCOME:", y)

        # 1) POINT EVENT STUDY (as before)
        event_cols = [k_to_col("evt", k) for k in ks]
        rhs_point = event_cols + CONTROLS + ["C(Country_Code)", "C(Year)"]
        m_point, used_point = fit_ols(df_es, y, rhs_point)

        tab = []
        for k in ks:
            term = k_to_col("evt", k)
            tab.append({
                "k": k,
                "term": term,
                "coef": float(m_point.params.get(term, np.nan)),
                "se": float(m_point.bse.get(term, np.nan)),
                "pval": float(m_point.pvalues.get(term, np.nan)),
            })
        tab = pd.DataFrame(tab).sort_values("k")
        print("POINT event-study (baseline k=-1 omitted):")
        print(tab.to_string(index=False))

        # Joint pretrend test (k=-5..-2)
        pre_terms = [k_to_col("evt", k) for k in range(-K_PRE, -1) if k != BASE]
        ft = joint_pretrend_test(m_point, pre_terms)
        if ft is not None:
            print("\nJOINT PRETREND TEST (k=-5..-2):")
            print("F-stat:", float(ft.fvalue), "p-value:", float(ft.pvalue))

        tab.to_csv(f"eventstudy_results_{y}.csv", index=False)

        # 2) BINNED EVENT STUDY (easier + more stable)
        bins = ["bin_pre_m5_m2", "bin_post_0_2", "bin_post_3_5", "bin_post_6_10"]
        rhs_bin = bins + CONTROLS + ["C(Country_Code)", "C(Year)"]
        m_bin, used_bin = fit_ols(df_es, y, rhs_bin)

        out_bin = []
        for b in bins:
            out_bin.append({
                "bin": b,
                "coef": float(m_bin.params.get(b, np.nan)),
                "se": float(m_bin.bse.get(b, np.nan)),
                "pval": float(m_bin.pvalues.get(b, np.nan)),
            })
        out_bin = pd.DataFrame(out_bin)
        print("\nBINNED effects (baseline is k=-1; bins are treated-period indicators):")
        print(out_bin.to_string(index=False))
        out_bin.to_csv(f"eventstudy_binned_{y}.csv", index=False)

    print("\nDONE.")

if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -------------------------
# CONFIG
# -------------------------
DATA_PATH = "../data/final_data/final_data.csv"
QAROT_DIR = "../data/qarot_data"
QAROT_FILE = None  # optional exact filename

YEAR_MIN = 1997
YEAR_MAX = 2015

K_PRE = 5
K_POST = 5
BASE = -1

# Cohort restriction so we can SEE both pre and post in 1997-2015
TREAT_MIN = YEAR_MIN + K_PRE   # 2002
TREAT_MAX = YEAR_MAX - K_POST  # 2010

CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]

# Candidate outcomes (script will run only those present)
CANDIDATE_OUTCOMES = [
    "PARL",
    "Human_Capital",
    "Life_Expectancy",
    "Female_Labor_Force",
    "Women_Business",
    "Women_Civil_Society",
    "Female_Adult_Literacy",
    "Fertility",
]

# -------------------------
# HELPERS
# -------------------------
def find_qarot_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path
    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")
    return os.path.join(folder, csvs[0])

def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]
    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")
    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year"})
    )

    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year_fallback"})
    )

    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])
    return out

def k_to_col(prefix: str, k: int) -> str:
    if k < 0:
        return f"{prefix}_m{abs(k)}"
    if k > 0:
        return f"{prefix}_p{k}"
    return f"{prefix}_0"

def make_event_core(df: pd.DataFrame):
    d = df.copy()
    d["ever_treated"] = d["treat_year"].notna().astype(int)
    d["rel_year"] = d["Year"] - d["treat_year"]
    return d

def restrict_to_cohorts(d: pd.DataFrame):
    """
    Keep:
    - never-treated (treat_year is NA), as controls
    - treated with treat_year in [TREAT_MIN, TREAT_MAX], so we can observe -5..+5
    and only keep treated obs within the event window.
    """
    treated_ok = d["treat_year"].between(TREAT_MIN, TREAT_MAX)
    keep = (d["ever_treated"] == 0) | treated_ok
    d = d[keep].copy()

    in_window = d["rel_year"].between(-K_PRE, K_POST)
    d = d[(d["ever_treated"] == 0) | in_window].copy()
    return d

def add_bins(d: pd.DataFrame):
    def treated_k_in(a, b):
        return ((d["ever_treated"] == 1) & (d["rel_year"].between(a, b))).astype(int)

    d["bin_pre"] = treated_k_in(-K_PRE, -2)
    d["bin_post_0_2"] = treated_k_in(0, 2)
    d["bin_post_3_5"] = treated_k_in(3, 5)
    return d

def fit_binned(df: pd.DataFrame, y: str):
    bins = ["bin_pre", "bin_post_0_2", "bin_post_3_5"]
    rhs = bins + CONTROLS + ["C(Country_Code)", "C(Year)"]
    formula = f"{y} ~ " + " + ".join(rhs)

    needed = [y, "Country_Code", "Year"] + bins + CONTROLS
    tmp = df.dropna(subset=needed).copy()

    m = smf.ols(formula=formula, data=tmp).fit(
        cov_type="cluster",
        cov_kwds={"groups": tmp["Country_Code"]}
    )

    out = []
    for b in bins:
        out.append({
            "bin": b,
            "coef": float(m.params.get(b, np.nan)),
            "se": float(m.bse.get(b, np.nan)),
            "pval": float(m.pvalues.get(b, np.nan)),
        })

    return pd.DataFrame(out), tmp

# -------------------------
# MAIN
# -------------------------
def main():
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    qarot_path = find_qarot_csv(QAROT_DIR, QAROT_FILE)
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    d = make_event_core(df)
    print("Countries in panel:", d["Country_Code"].nunique())
    print("Treated countries (any treat_year):", d.loc[d["ever_treated"] == 1, "Country_Code"].nunique())
    print("Cohort restriction treat_year in:", TREAT_MIN, "-", TREAT_MAX)

    d = restrict_to_cohorts(d)
    d = add_bins(d)

    # Save cohort sample for transparency
    d.to_csv("merged_panel_qarot_cohorts_1997_2015_k5.csv", index=False)
    print("Saved:", "merged_panel_qarot_cohorts_1997_2015_k5.csv")

    # Run outcomes
    results = []
    for y in CANDIDATE_OUTCOMES:
        if y not in d.columns:
            continue

        tab, used = fit_binned(d, y)
        tab.insert(0, "outcome", y)
        tab["n_obs"] = used.shape[0]
        tab["n_countries"] = used["Country_Code"].nunique()
        tab["year_min"] = used["Year"].min()
        tab["year_max"] = used["Year"].max()

        print("\n==============================")
        print("OUTCOME:", y)
        print(tab.to_string(index=False))

        tab.to_csv(f"binned_cohort_results_{y}.csv", index=False)
        results.append(tab)

    if results:
        all_res = pd.concat(results, ignore_index=True)
        all_res.to_csv("binned_cohort_results_ALL.csv", index=False)
        print("\nSaved:", "binned_cohort_results_ALL.csv")

    print("\nDONE.")

if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA_PATH = "../data/final_data/final_data.csv"
QAROT_DIR = "../data/qarot_data"
QAROT_FILE = None

YEAR_MIN, YEAR_MAX = 1997, 2015
K_PRE, K_POST = 5, 5

TREAT_MIN = YEAR_MIN + K_PRE   # 2002
TREAT_MAX = YEAR_MAX - K_POST  # 2010

CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]
CANDIDATE_OUTCOMES = [
    "PARL",
    "Human_Capital",
    "Life_Expectancy",
    "Female_Labor_Force",
    "Women_Business",
    "Women_Civil_Society",
    "Female_Adult_Literacy",
    "Fertility",
]

def find_qarot_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path
    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")
    return os.path.join(folder, csvs[0])

def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]
    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")
    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year"})
    )
    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year_fallback"})
    )
    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])
    return out

def make_core(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["ever_treated"] = d["treat_year"].notna().astype(int)
    d["rel_year"] = d["Year"] - d["treat_year"]
    return d

def restrict_to_cohorts(d: pd.DataFrame) -> pd.DataFrame:
    treated_ok = d["treat_year"].between(TREAT_MIN, TREAT_MAX)
    d = d[(d["ever_treated"] == 0) | treated_ok].copy()

    in_window = d["rel_year"].between(-K_PRE, K_POST)
    d = d[(d["ever_treated"] == 0) | in_window].copy()
    return d

def add_bins(d: pd.DataFrame) -> pd.DataFrame:
    def treated_k_in(a, b):
        return ((d["ever_treated"] == 1) & (d["rel_year"].between(a, b))).astype(int)

    d["bin_pre"] = treated_k_in(-K_PRE, -2)
    d["bin_post_0_2"] = treated_k_in(0, 2)
    d["bin_post_3_5"] = treated_k_in(3, 5)
    return d

def bin_diagnostics(d: pd.DataFrame):
    bins = ["bin_pre", "bin_post_0_2", "bin_post_3_5"]
    print("\n=== BIN DIAGNOSTICS (treated observations only) ===")
    treated = d[d["ever_treated"] == 1].copy()
    for b in bins:
        n_obs = int(treated[b].sum())
        # number of countries that contribute at least one obs in this bin
        n_c = treated.loc[treated[b] == 1, "Country_Code"].nunique()
        years = treated.loc[treated[b] == 1, "Year"]
        y_min = int(years.min()) if len(years) else None
        y_max = int(years.max()) if len(years) else None
        print(f"{b}: treated obs={n_obs}, treated countries={n_c}, year_range={y_min}-{y_max}")

def fit_binned(d: pd.DataFrame, y: str):
    bins = ["bin_pre", "bin_post_0_2", "bin_post_3_5"]
    rhs = bins + CONTROLS + ["C(Country_Code)", "C(Year)"]
    formula = f"{y} ~ " + " + ".join(rhs)

    needed = [y, "Country_Code", "Year"] + bins + CONTROLS
    tmp = d.dropna(subset=needed).copy()

    m = smf.ols(formula=formula, data=tmp).fit(
        cov_type="cluster",
        cov_kwds={"groups": tmp["Country_Code"]}
    )

    out = []
    for b in bins:
        out.append({
            "bin": b,
            "coef": float(m.params[b]) if b in m.params.index else np.nan,
            "se": float(m.bse[b]) if b in m.bse.index else np.nan,
            "pval": float(m.pvalues[b]) if b in m.pvalues.index else np.nan,
        })

    res = pd.DataFrame(out)
    return res, tmp

def main():
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    qarot_path = find_qarot_csv(QAROT_DIR, QAROT_FILE)
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    d = make_core(df)
    d = restrict_to_cohorts(d)
    d = add_bins(d)

    print("Countries in cohort sample:", d["Country_Code"].nunique())
    print("Treated countries in cohort sample:", d.loc[d["ever_treated"] == 1, "Country_Code"].nunique())
    bin_diagnostics(d)

    d.to_csv("merged_panel_qarot_cohorts_1997_2015_k5.csv", index=False)

    for y in CANDIDATE_OUTCOMES:
        if y not in d.columns:
            continue
        tab, used = fit_binned(d, y)
        tab.insert(0, "outcome", y)
        tab["n_obs"] = used.shape[0]
        tab["n_countries"] = used["Country_Code"].nunique()

        print("\n==============================")
        print("OUTCOME:", y)
        print(tab.to_string(index=False))

    print("\nDONE.")

if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA_PATH = "../data/final_data/final_data.csv"
QAROT_DIR = "../data/qarot_data"
QAROT_FILE = None  # optional exact filename

YEAR_MIN, YEAR_MAX = 1997, 2022

# keep cohorts where we can see pre and at least some post
K_PRE = 5
TREAT_MIN = YEAR_MIN + K_PRE   # 2002
TREAT_MAX = 2010               # keep as before (clean cohorts)

CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]

CANDIDATE_OUTCOMES = [
    "PARL",
    "Human_Capital",
    "Life_Expectancy",
    "Female_Labor_Force",
    "Women_Business",
    "Women_Civil_Society",
    "Female_Adult_Literacy",
    "Fertility",
]

BINS = {
    "bin_pre_m5_m2": (-5, -2),
    "bin_post_0_2": (0, 2),
    "bin_post_3_5": (3, 5),
    "bin_post_6_10": (6, 10),
    "bin_post_11_12": (11, 12),
}

def find_qarot_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path
    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")
    return os.path.join(folder, csvs[0])

def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]
    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")
    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year"})
    )
    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year_fallback"})
    )
    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])
    return out

def make_core(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["ever_treated"] = d["treat_year"].notna().astype(int)
    d["rel_year"] = d["Year"] - d["treat_year"]
    return d

def restrict_to_cohorts(d: pd.DataFrame) -> pd.DataFrame:
    treated_ok = d["treat_year"].between(TREAT_MIN, TREAT_MAX)
    d = d[(d["ever_treated"] == 0) | treated_ok].copy()
    return d

def add_bins(d: pd.DataFrame) -> pd.DataFrame:
    def treated_k_in(a, b):
        return ((d["ever_treated"] == 1) & (d["rel_year"].between(a, b))).astype(int)

    for name, (a, b) in BINS.items():
        d[name] = treated_k_in(a, b)
    return d

def outcome_specific_bin_diag(d: pd.DataFrame, y: str, bin_names: list):
    # This is the ACTUAL regression sample (dropna on y+controls+bins+FE keys)
    needed = [y, "Country_Code", "Year"] + CONTROLS + bin_names
    tmp = d.dropna(subset=needed).copy()

    treated = tmp[tmp["ever_treated"] == 1].copy()
    out = []
    for b in bin_names:
        n_obs = int(treated[b].sum())
        n_c = treated.loc[treated[b] == 1, "Country_Code"].nunique()
        yrs = treated.loc[treated[b] == 1, "Year"]
        y_min = int(yrs.min()) if len(yrs) else None
        y_max = int(yrs.max()) if len(yrs) else None
        out.append((b, n_obs, n_c, y_min, y_max))

    return tmp, out

def fit_binned(tmp: pd.DataFrame, y: str, bin_names: list):
    rhs = bin_names + CONTROLS + ["C(Country_Code)", "C(Year)"]
    formula = f"{y} ~ " + " + ".join(rhs)

    m = smf.ols(formula=formula, data=tmp).fit(
        cov_type="cluster",
        cov_kwds={"groups": tmp["Country_Code"]}
    )

    rows = []
    for b in bin_names:
        rows.append({
            "bin": b,
            "coef": float(m.params[b]) if b in m.params.index else np.nan,
            "se": float(m.bse[b]) if b in m.bse.index else np.nan,
            "pval": float(m.pvalues[b]) if b in m.pvalues.index else np.nan,
            "in_model": b in m.params.index
        })
    return pd.DataFrame(rows), m

def main():
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    qarot_path = find_qarot_csv(QAROT_DIR, QAROT_FILE)
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    d = make_core(df)
    d = restrict_to_cohorts(d)
    d = add_bins(d)

    print("Countries in sample:", d["Country_Code"].nunique())
    print("Treated countries in sample:", d.loc[d["ever_treated"] == 1, "Country_Code"].nunique())
    print("Cohorts treat_year:", TREAT_MIN, "-", TREAT_MAX)
    print("Years:", YEAR_MIN, "-", YEAR_MAX)

    bin_names = list(BINS.keys())

    all_rows = []
    for y in CANDIDATE_OUTCOMES:
        if y not in d.columns:
            continue

        tmp, diag = outcome_specific_bin_diag(d, y, bin_names)

        print("\n==============================")
        print("OUTCOME:", y)
        print("Regression sample: n_obs =", tmp.shape[0], ", n_countries =", tmp["Country_Code"].nunique())
        print("Treated countries in regression sample:", tmp.loc[tmp["ever_treated"] == 1, "Country_Code"].nunique())
        print("Bin fill AFTER dropna (treated only):")
        for (b, n_obs, n_c, y_min, y_max) in diag:
            print(f"  {b}: treated obs={n_obs}, treated countries={n_c}, years={y_min}-{y_max}")

        tab, _ = fit_binned(tmp, y, bin_names)
        tab.insert(0, "outcome", y)
        tab["n_obs"] = tmp.shape[0]
        tab["n_countries"] = tmp["Country_Code"].nunique()
        print("\nBinned coefficients:")
        print(tab.to_string(index=False))

        all_rows.append(tab)

    if all_rows:
        out = pd.concat(all_rows, ignore_index=True)
        out.to_csv("binned_cohort_results_1997_2022_ALL.csv", index=False)
        print("\nSaved: binned_cohort_results_1997_2022_ALL.csv")

    print("\nDONE.")

if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA_PATH = "../data/final_data/final_data.csv"
QAROT_DIR = "../data/qarot_data"
QAROT_FILE = None  # optional exact filename

YEAR_MIN, YEAR_MAX = 1997, 2022

K_PRE = 5
K_POST_MAX = 12   # window ends at +12
TREAT_MIN = YEAR_MIN + K_PRE   # 2002
TREAT_MAX = 2010               # clean cohorts (same as before)

CONTROLS = ["GDP", "Political_Stability", "Fertility", "Corruption"]

OUTCOMES_TO_PRINT = ["PARL", "Human_Capital"]  # keep minimal; you can add others

BINS = {
    "bin_pre_m5_m2": (-5, -2),
    "bin_post_0_2": (0, 2),
    "bin_post_3_5": (3, 5),
    "bin_post_6_10": (6, 10),
    "bin_post_11_12": (11, 12),
}

def find_qarot_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path
    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")
    return os.path.join(folder, csvs[0])

def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]
    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")
    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year"})
    )
    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
         .dropna()
         .groupby("code", as_index=False)["year"].min()
         .rename(columns={"year": "treat_year_fallback"})
    )
    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])
    return out

def make_core(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["ever_treated"] = d["treat_year"].notna().astype(int)
    d["rel_year"] = d["Year"] - d["treat_year"]
    return d

def restrict_to_cohorts(d: pd.DataFrame) -> pd.DataFrame:
    treated_ok = d["treat_year"].between(TREAT_MIN, TREAT_MAX)
    d = d[(d["ever_treated"] == 0) | treated_ok].copy()
    return d

def restrict_treated_to_window(d: pd.DataFrame) -> pd.DataFrame:
    # Keep all never-treated years;
    # For treated, keep only rel_year in [-5, +12] so baseline is not contaminated by k>12.
    in_window = d["rel_year"].between(-K_PRE, K_POST_MAX)
    d = d[(d["ever_treated"] == 0) | in_window].copy()
    return d

def add_bins(d: pd.DataFrame) -> pd.DataFrame:
    def treated_k_in(a, b):
        return ((d["ever_treated"] == 1) & (d["rel_year"].between(a, b))).astype(int)
    for name, (a, b) in BINS.items():
        d[name] = treated_k_in(a, b)
    return d

def outcome_specific_bin_diag(d: pd.DataFrame, y: str, bin_names: list):
    needed = [y, "Country_Code", "Year"] + CONTROLS + bin_names
    tmp = d.dropna(subset=needed).copy()

    treated = tmp[tmp["ever_treated"] == 1].copy()
    diag = []
    for b in bin_names:
        n_obs = int(treated[b].sum())
        n_c = treated.loc[treated[b] == 1, "Country_Code"].nunique()
        yrs = treated.loc[treated[b] == 1, "Year"]
        y_min = int(yrs.min()) if len(yrs) else None
        y_max = int(yrs.max()) if len(yrs) else None
        diag.append((b, n_obs, n_c, y_min, y_max))
    return tmp, diag

def fit_binned(tmp: pd.DataFrame, y: str, bin_names: list):
    rhs = bin_names + CONTROLS + ["C(Country_Code)", "C(Year)"]
    formula = f"{y} ~ " + " + ".join(rhs)

    m = smf.ols(formula=formula, data=tmp).fit(
        cov_type="cluster",
        cov_kwds={"groups": tmp["Country_Code"]}
    )

    rows = []
    for b in bin_names:
        rows.append({
            "bin": b,
            "coef": float(m.params[b]) if b in m.params.index else np.nan,
            "se": float(m.bse[b]) if b in m.bse.index else np.nan,
            "pval": float(m.pvalues[b]) if b in m.pvalues.index else np.nan,
            "in_model": b in m.params.index
        })
    return pd.DataFrame(rows)

def main():
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    qarot_path = find_qarot_csv(QAROT_DIR, QAROT_FILE)
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    d = make_core(df)
    d = restrict_to_cohorts(d)

    print("Countries in sample (before window):", d["Country_Code"].nunique())
    print("Treated countries in sample (before window):", d.loc[d["ever_treated"] == 1, "Country_Code"].nunique())

    # IMPORTANT FIX
    d = restrict_treated_to_window(d)

    print("Countries in sample (after window):", d["Country_Code"].nunique())
    print("Treated countries in sample (after window):", d.loc[d["ever_treated"] == 1, "Country_Code"].nunique())
    print("Cohorts treat_year:", TREAT_MIN, "-", TREAT_MAX)
    print("Years:", YEAR_MIN, "-", YEAR_MAX)
    print("Event window for treated: rel_year in [-5, +12]")

    d = add_bins(d)

    d.to_csv("merged_panel_qarot_cohorts_1997_2022_window_m5_p12.csv", index=False)
    print("Saved: merged_panel_qarot_cohorts_1997_2022_window_m5_p12.csv")

    bin_names = list(BINS.keys())

    for y in OUTCOMES_TO_PRINT:
        if y not in d.columns:
            continue

        tmp, diag = outcome_specific_bin_diag(d, y, bin_names)

        print("\n==============================")
        print("OUTCOME:", y)
        print("Regression sample: n_obs =", tmp.shape[0], ", n_countries =", tmp["Country_Code"].nunique())
        print("Treated countries in regression sample:", tmp.loc[tmp["ever_treated"] == 1, "Country_Code"].nunique())
        print("Bin fill AFTER dropna (treated only):")
        for (b, n_obs, n_c, y_min, y_max) in diag:
            print(f"  {b}: treated obs={n_obs}, treated countries={n_c}, years={y_min}-{y_max}")

        tab = fit_binned(tmp, y, bin_names)
        tab.insert(0, "outcome", y)
        tab["n_obs"] = tmp.shape[0]
        tab["n_countries"] = tmp["Country_Code"].nunique()
        print("\nBinned coefficients:")
        print(tab.to_string(index=False))

    print("\nDONE.")

if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
"""
v6: Point event-study (k=-5..+12, baseline k=-1 omitted) for:
- Fertility
- Female_Labor_Force

Design:
- Cohorts: treat_year in [2002, 2010] for treated countries; never-treated kept
- Event window for treated: rel_year in [-5, +12] (prevents baseline contamination)
- FE: Country + Year
- SE: clustered by Country
- Controls: GDP, Political_Stability, Fertility, Corruption (but outcome excluded from controls automatically)
- Joint pretrend F-test for leads k=-5..-2
- Also prints binned summary (same bins as before) as a compact “story” table

Inputs:
- Panel: ../data/final_data/final_data.csv  (sep=';', decimal=',')
- QAROT: ../data/qarot_data/*.csv  (or ./qarot_data/*.csv as fallback)
Outputs:
- merged_panel_qarot_cohorts_1997_2022_window_m5_p12_v6.csv
- eventstudy_point_results_<OUTCOME>.csv
- eventstudy_binned_results_<OUTCOME>.csv
"""

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------
# Paths / settings
# -----------------------
DATA_PATH = "../data/final_data/final_data.csv"

# Try these folders in order (you used both patterns earlier)
QAROT_DIR_CANDIDATES = ["../data/qarot_data", "qarot_data"]
QAROT_FILE = None  # optional exact filename; if None -> auto-pick first csv in folder

YEAR_MIN, YEAR_MAX = 1997, 2022

K_MIN, K_MAX = -5, 12          # point event-study window
BASELINE_K = -1                # omitted baseline
TREAT_MIN, TREAT_MAX = 2002, 2010  # cohort restriction

CONTROLS_DEFAULT = ["GDP", "Political_Stability", "Fertility", "Corruption"]

OUTCOMES = ["Fertility", "Female_Labor_Force"]

# bins for an easy summary table (baseline is still k=-1 omitted in point spec)
BINS = {
    "bin_pre_m5_m2": (-5, -2),
    "bin_post_0_2": (0, 2),
    "bin_post_3_5": (3, 5),
    "bin_post_6_10": (6, 10),
    "bin_post_11_12": (11, 12),
}


# -----------------------
# Helpers
# -----------------------
def find_first_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path
    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")
    return os.path.join(folder, csvs[0])


def resolve_qarot_path() -> str:
    for d in QAROT_DIR_CANDIDATES:
        if os.path.isdir(d):
            try:
                return find_first_csv(d, QAROT_FILE)
            except FileNotFoundError:
                continue
    raise FileNotFoundError(
        f"Could not find QAROT csv. Tried folders: {QAROT_DIR_CANDIDATES}"
    )


def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    """
    Build treat_year as the earliest year when quota is implemented.
    Uses:
      - first.implement.flag == 1 (preferred)
      - implemented.quota == 1 (fallback)
    """
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]

    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")

    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
        .dropna()
        .groupby("code", as_index=False)["year"].min()
        .rename(columns={"year": "treat_year"})
    )

    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
        .dropna()
        .groupby("code", as_index=False)["year"].min()
        .rename(columns={"year": "treat_year_fallback"})
    )

    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])
    return out


def term_name_for_k(k: int) -> str:
    """Safe patsy term name (no minus sign)."""
    if k < 0:
        return f"evt_m{abs(k)}"
    if k > 0:
        return f"evt_p{k}"
    return "evt_0"


def add_point_dummies(d: pd.DataFrame) -> tuple[pd.DataFrame, list[int], list[str]]:
    """
    Create event-time dummies for treated only:
      D_k = 1{ever_treated=1 AND rel_year == k}
    Omit baseline k=-1 from the formula.
    """
    ks = [k for k in range(K_MIN, K_MAX + 1) if k != BASELINE_K]
    cols = []
    for k in ks:
        col = term_name_for_k(k)
        d[col] = ((d["ever_treated"] == 1) & (d["rel_year"] == k)).astype(int)
        cols.append(col)
    return d, ks, cols


def add_bins(d: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    bin_cols = []
    for name, (a, b) in BINS.items():
        d[name] = ((d["ever_treated"] == 1) & (d["rel_year"].between(a, b))).astype(int)
        bin_cols.append(name)
    return d, bin_cols


def restrict_to_cohorts_and_window(d: pd.DataFrame) -> pd.DataFrame:
    # cohorts: treated countries must have treat_year in [TREAT_MIN, TREAT_MAX]
    treated_ok = d["treat_year"].between(TREAT_MIN, TREAT_MAX)
    d = d[(d["ever_treated"] == 0) | treated_ok].copy()

    # window: for treated keep only rel_year in [K_MIN, K_MAX]
    in_window = d["rel_year"].between(K_MIN, K_MAX)
    d = d[(d["ever_treated"] == 0) | in_window].copy()

    return d


def choose_controls(outcome: str, available_cols: list[str]) -> list[str]:
    # Remove outcome from controls if present
    ctrls = [c for c in CONTROLS_DEFAULT if c != outcome and c in available_cols]
    return ctrls


def fit_ols_cluster(formula: str, df: pd.DataFrame):
    return smf.ols(formula=formula, data=df).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["Country_Code"]},
    )


def extract_point_table(model, ks: list[int]) -> pd.DataFrame:
    rows = []
    for k in ks:
        col = term_name_for_k(k)
        rows.append(
            {
                "k": k,
                "term": col,
                "coef": float(model.params[col]) if col in model.params.index else np.nan,
                "se": float(model.bse[col]) if col in model.bse.index else np.nan,
                "pval": float(model.pvalues[col]) if col in model.pvalues.index else np.nan,
                "in_model": col in model.params.index,
            }
        )
    return pd.DataFrame(rows).sort_values("k")


def joint_pretrend_test(model, leads_ks: list[int]):
    """
    Joint F-test that coefficients on lead dummies (k in leads_ks) are all zero.
    """
    terms = [term_name_for_k(k) for k in leads_ks]
    terms = [t for t in terms if t in model.params.index]
    if not terms:
        return None

    hyp = " = 0, ".join(terms) + " = 0"
    ftest = model.f_test(hyp)
    return float(ftest.fvalue), float(ftest.pvalue), terms


def fit_event_study_point(df: pd.DataFrame, outcome: str):
    # controls (exclude outcome)
    controls = choose_controls(outcome, df.columns.tolist())

    # create point dummies
    d = df.copy()
    d, ks, event_cols = add_point_dummies(d)

    needed = [outcome, "Country_Code", "Year"] + controls + event_cols
    tmp = d.dropna(subset=needed).copy()

    rhs = event_cols + controls + ["C(Country_Code)", "C(Year)"]
    formula = f"{outcome} ~ " + " + ".join(rhs)
    model = fit_ols_cluster(formula, tmp)

    tab = extract_point_table(model, ks)

    # joint pretrend: leads k=-5..-2
    leads = [k for k in range(-5, -1) if k != BASELINE_K]  # -5,-4,-3,-2
    jt = joint_pretrend_test(model, leads)

    diag = {
        "outcome": outcome,
        "n_obs": int(tmp.shape[0]),
        "n_countries": int(tmp["Country_Code"].nunique()),
        "treated_countries_in_reg": int(tmp.loc[tmp["ever_treated"] == 1, "Country_Code"].nunique()),
        "year_min": int(tmp["Year"].min()),
        "year_max": int(tmp["Year"].max()),
        "controls": controls,
    }

    return tab, jt, diag


def fit_event_study_binned(df: pd.DataFrame, outcome: str):
    controls = choose_controls(outcome, df.columns.tolist())

    d = df.copy()
    d, bin_cols = add_bins(d)

    needed = [outcome, "Country_Code", "Year"] + controls + bin_cols
    tmp = d.dropna(subset=needed).copy()

    rhs = bin_cols + controls + ["C(Country_Code)", "C(Year)"]
    formula = f"{outcome} ~ " + " + ".join(rhs)
    model = fit_ols_cluster(formula, tmp)

    rows = []
    for b in bin_cols:
        rows.append(
            {
                "outcome": outcome,
                "bin": b,
                "coef": float(model.params[b]) if b in model.params.index else np.nan,
                "se": float(model.bse[b]) if b in model.bse.index else np.nan,
                "pval": float(model.pvalues[b]) if b in model.pvalues.index else np.nan,
                "in_model": b in model.params.index,
                "n_obs": int(tmp.shape[0]),
                "n_countries": int(tmp["Country_Code"].nunique()),
                "treated_countries_in_reg": int(tmp.loc[tmp["ever_treated"] == 1, "Country_Code"].nunique()),
            }
        )

    # outcome-specific treated-only bin fill AFTER dropna
    treated = tmp[tmp["ever_treated"] == 1].copy()
    bin_fill = []
    for b in bin_cols:
        n_obs = int(treated[b].sum())
        n_c = int(treated.loc[treated[b] == 1, "Country_Code"].nunique())
        yrs = treated.loc[treated[b] == 1, "Year"]
        y_min = int(yrs.min()) if len(yrs) else None
        y_max = int(yrs.max()) if len(yrs) else None
        bin_fill.append((b, n_obs, n_c, y_min, y_max))

    return pd.DataFrame(rows), bin_fill


# -----------------------
# Main
# -----------------------
def main():
    # Read main panel
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    # Read QAROT + merge treat_year
    qarot_path = resolve_qarot_path()
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    # Core vars
    df["ever_treated"] = df["treat_year"].notna().astype(int)
    df["rel_year"] = df["Year"] - df["treat_year"]

    # Restrict cohorts and treated window
    d = restrict_to_cohorts_and_window(df)

    print("Countries in sample:", d["Country_Code"].nunique())
    print("Treated countries in sample:", d.loc[d["ever_treated"] == 1, "Country_Code"].nunique())
    print("Cohorts treat_year:", TREAT_MIN, "-", TREAT_MAX)
    print("Years:", YEAR_MIN, "-", YEAR_MAX)
    print(f"Event window for treated: rel_year in [{K_MIN}, +{K_MAX}] (baseline k={BASELINE_K} omitted)")

    # Save merged dataset for reproducibility
    out_panel = "merged_panel_qarot_cohorts_1997_2022_window_m5_p12_v6.csv"
    d.to_csv(out_panel, index=False)
    print("Saved:", out_panel)

    for outcome in OUTCOMES:
        if outcome not in d.columns:
            print("\nSKIP (missing outcome column):", outcome)
            continue

        print("\n==============================")
        print("POINT EVENT STUDY:", outcome)

        tab, jt, diag = fit_event_study_point(d, outcome)
        print("N obs:", diag["n_obs"], "countries:", diag["n_countries"], "treated countries in reg:", diag["treated_countries_in_reg"])
        print("Years:", diag["year_min"], "-", diag["year_max"])
        print("Controls used:", diag["controls"])
        print("\nPOINT effects (baseline k=-1 omitted):")
        print(tab[["k", "term", "coef", "se", "pval"]].to_string(index=False))

        if jt is None:
            print("\nJOINT PRETREND TEST (k=-5..-2): not available (missing lead terms in model)")
        else:
            fstat, pval, used_terms = jt
            print("\nJOINT PRETREND TEST (k=-5..-2):")
            print("Terms used:", used_terms)
            print("F-stat:", fstat, "p-value:", pval)

        # Save point table
        point_out = f"eventstudy_point_results_{outcome}.csv"
        tab.to_csv(point_out, index=False)
        print("Saved:", point_out)

        # Binned summary (for a clean “story” table)
        print("\n------------------------------")
        print("BINNED SUMMARY:", outcome)
        btab, bfill = fit_event_study_binned(d, outcome)
        print("Bin fill AFTER dropna (treated only):")
        for (b, n_obs, n_c, y_min, y_max) in bfill:
            print(f"  {b}: treated obs={n_obs}, treated countries={n_c}, years={y_min}-{y_max}")

        print("\nBinned coefficients:")
        print(btab[["outcome", "bin", "coef", "se", "pval", "n_obs", "n_countries"]].to_string(index=False))

        binned_out = f"eventstudy_binned_results_{outcome}.csv"
        btab.to_csv(binned_out, index=False)
        print("Saved:", binned_out)

    print("\nDONE.")


if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
"""
v7: Make event-study PACK for dissertation:
- Point event-study (k=-5..+12, baseline k=-1 omitted), Country+Year FE, cluster by country
- Saves: point tables CSV + binned tables CSV + PNG plots with 95% CI
- Works on the same cohort/window logic as v6.

Inputs:
- ../data/final_data/final_data.csv (sep=';', decimal=',')
- QAROT in ../data/qarot_data/*.csv or ./qarot_data/*.csv

Outputs:
- eventstudy_point_results_<OUTCOME>.csv
- eventstudy_binned_results_<OUTCOME>.csv
- eventstudy_plot_<OUTCOME>.png
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

DATA_PATH = "../data/final_data/final_data.csv"
QAROT_DIR_CANDIDATES = ["../data/qarot_data", "qarot_data"]
QAROT_FILE = None

YEAR_MIN, YEAR_MAX = 1997, 2022
K_MIN, K_MAX = -5, 12
BASELINE_K = -1
TREAT_MIN, TREAT_MAX = 2002, 2010

CONTROLS_DEFAULT = ["GDP", "Political_Stability", "Fertility", "Corruption"]

# What to pack (you can add/remove)
OUTCOMES = [
    "PARL",              # to show first-stage in the same sample
    "Human_Capital",
    "Life_Expectancy",
    "Fertility",
    "Female_Labor_Force",
    "Women_Business",
    "Women_Civil_Society",
]

BINS = {
    "bin_pre_m5_m2": (-5, -2),
    "bin_post_0_2": (0, 2),
    "bin_post_3_5": (3, 5),
    "bin_post_6_10": (6, 10),
    "bin_post_11_12": (11, 12),
}


def find_first_csv(folder: str, filename: str | None) -> str:
    if filename:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"QAROT file not found: {path}")
        return path
    csvs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])
    if not csvs:
        raise FileNotFoundError(f"No .csv files found in folder: {folder}")
    return os.path.join(folder, csvs[0])


def resolve_qarot_path() -> str:
    for d in QAROT_DIR_CANDIDATES:
        if os.path.isdir(d):
            try:
                return find_first_csv(d, QAROT_FILE)
            except FileNotFoundError:
                continue
    raise FileNotFoundError(f"Could not find QAROT csv. Tried folders: {QAROT_DIR_CANDIDATES}")


def build_treat_year(qarot: pd.DataFrame) -> pd.DataFrame:
    q = qarot.copy()
    q.columns = [c.strip().lower() for c in q.columns]

    if "code" not in q.columns or "year" not in q.columns:
        raise ValueError("QAROT expected columns 'code' and 'year' not found.")

    q["year"] = pd.to_numeric(q["year"], errors="coerce").astype("Int64")

    for col in ["first.implement.flag", "implemented.quota"]:
        if col in q.columns:
            q[col] = pd.to_numeric(q[col], errors="coerce")
        else:
            q[col] = np.nan

    treat1 = (
        q.loc[q["first.implement.flag"] == 1, ["code", "year"]]
        .dropna()
        .groupby("code", as_index=False)["year"].min()
        .rename(columns={"year": "treat_year"})
    )

    treat2 = (
        q.loc[q["implemented.quota"] == 1, ["code", "year"]]
        .dropna()
        .groupby("code", as_index=False)["year"].min()
        .rename(columns={"year": "treat_year_fallback"})
    )

    out = pd.merge(treat1, treat2, on="code", how="outer")
    out["treat_year"] = out["treat_year"].fillna(out["treat_year_fallback"])
    out = out.drop(columns=["treat_year_fallback"])
    return out


def term_name_for_k(k: int) -> str:
    if k < 0:
        return f"evt_m{abs(k)}"
    if k > 0:
        return f"evt_p{k}"
    return "evt_0"


def add_point_dummies(d: pd.DataFrame):
    ks = [k for k in range(K_MIN, K_MAX + 1) if k != BASELINE_K]
    cols = []
    for k in ks:
        col = term_name_for_k(k)
        d[col] = ((d["ever_treated"] == 1) & (d["rel_year"] == k)).astype(int)
        cols.append(col)
    return d, ks, cols


def add_bins(d: pd.DataFrame):
    bin_cols = []
    for name, (a, b) in BINS.items():
        d[name] = ((d["ever_treated"] == 1) & (d["rel_year"].between(a, b))).astype(int)
        bin_cols.append(name)
    return d, bin_cols


def restrict_to_cohorts_and_window(d: pd.DataFrame) -> pd.DataFrame:
    treated_ok = d["treat_year"].between(TREAT_MIN, TREAT_MAX)
    d = d[(d["ever_treated"] == 0) | treated_ok].copy()

    in_window = d["rel_year"].between(K_MIN, K_MAX)
    d = d[(d["ever_treated"] == 0) | in_window].copy()
    return d


def choose_controls(outcome: str, available_cols: list[str]) -> list[str]:
    return [c for c in CONTROLS_DEFAULT if c != outcome and c in available_cols]


def fit_ols_cluster(formula: str, df: pd.DataFrame):
    return smf.ols(formula=formula, data=df).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["Country_Code"]},
    )


def extract_point_table(model, ks: list[int]) -> pd.DataFrame:
    rows = []
    for k in ks:
        col = term_name_for_k(k)
        coef = float(model.params[col]) if col in model.params.index else np.nan
        se = float(model.bse[col]) if col in model.bse.index else np.nan
        pval = float(model.pvalues[col]) if col in model.pvalues.index else np.nan
        rows.append({"k": k, "term": col, "coef": coef, "se": se, "pval": pval})
    return pd.DataFrame(rows).sort_values("k")


def joint_pretrend_test(model):
    leads = [-5, -4, -3, -2]
    terms = [term_name_for_k(k) for k in leads if term_name_for_k(k) in model.params.index]
    if not terms:
        return None
    hyp = " = 0, ".join(terms) + " = 0"
    ftest = model.f_test(hyp)
    return float(ftest.fvalue), float(ftest.pvalue), terms


def point_event_study(df: pd.DataFrame, outcome: str):
    controls = choose_controls(outcome, df.columns.tolist())
    d = df.copy()
    d, ks, event_cols = add_point_dummies(d)

    needed = [outcome, "Country_Code", "Year"] + controls + event_cols
    tmp = d.dropna(subset=needed).copy()

    rhs = event_cols + controls + ["C(Country_Code)", "C(Year)"]
    formula = f"{outcome} ~ " + " + ".join(rhs)
    model = fit_ols_cluster(formula, tmp)

    tab = extract_point_table(model, ks)
    jt = joint_pretrend_test(model)

    diag = {
        "outcome": outcome,
        "n_obs": int(tmp.shape[0]),
        "n_countries": int(tmp["Country_Code"].nunique()),
        "treated_countries_in_reg": int(tmp.loc[tmp["ever_treated"] == 1, "Country_Code"].nunique()),
        "year_min": int(tmp["Year"].min()),
        "year_max": int(tmp["Year"].max()),
        "controls": controls,
    }
    return tab, jt, diag


def binned_event_study(df: pd.DataFrame, outcome: str):
    controls = choose_controls(outcome, df.columns.tolist())
    d = df.copy()
    d, bin_cols = add_bins(d)

    needed = [outcome, "Country_Code", "Year"] + controls + bin_cols
    tmp = d.dropna(subset=needed).copy()

    rhs = bin_cols + controls + ["C(Country_Code)", "C(Year)"]
    formula = f"{outcome} ~ " + " + ".join(rhs)
    model = fit_ols_cluster(formula, tmp)

    rows = []
    for b in bin_cols:
        rows.append({
            "outcome": outcome,
            "bin": b,
            "coef": float(model.params[b]) if b in model.params.index else np.nan,
            "se": float(model.bse[b]) if b in model.bse.index else np.nan,
            "pval": float(model.pvalues[b]) if b in model.pvalues.index else np.nan,
            "n_obs": int(tmp.shape[0]),
            "n_countries": int(tmp["Country_Code"].nunique()),
        })
    return pd.DataFrame(rows)


def plot_event_study(tab: pd.DataFrame, outcome: str, pretrend_info=None):
    # 95% CI
    x = tab["k"].values
    y = tab["coef"].values
    se = tab["se"].values
    ci = 1.96 * se

    plt.figure(figsize=(9, 5))
    plt.axhline(0, linewidth=1)
    plt.axvline(-1, linestyle="--", linewidth=1)  # baseline marker (omitted)
    plt.errorbar(x, y, yerr=ci, fmt="o", capsize=3)
    plt.title(f"Event-study: {outcome} (baseline k=-1 omitted)")
    plt.xlabel("Event time k (years relative to quota adoption)")
    plt.ylabel("Coefficient (vs k=-1)")

    if pretrend_info is not None:
        fstat, pval, _ = pretrend_info
        plt.text(0.02, 0.02, f"Joint pretrend (k=-5..-2): p={pval:.3f}", transform=plt.gca().transAxes)

    out = f"eventstudy_plot_{outcome}.png"
    plt.tight_layout()
    plt.savefig(out, dpi=150)
    plt.close()
    return out


def main():
    df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)
    df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

    qarot_path = resolve_qarot_path()
    print("Using QAROT file:", qarot_path)
    qarot = pd.read_csv(qarot_path)

    treat = build_treat_year(qarot).rename(columns={"code": "Country_Code"})
    df = df.merge(treat, on="Country_Code", how="left")

    df["ever_treated"] = df["treat_year"].notna().astype(int)
    df["rel_year"] = df["Year"] - df["treat_year"]

    d = restrict_to_cohorts_and_window(df)

    print("Countries in sample:", d["Country_Code"].nunique())
    print("Treated countries in sample:", d.loc[d["ever_treated"] == 1, "Country_Code"].nunique())
    print("Cohorts treat_year:", TREAT_MIN, "-", TREAT_MAX)
    print("Years:", YEAR_MIN, "-", YEAR_MAX)
    print(f"Event window for treated: rel_year in [{K_MIN}, +{K_MAX}] (baseline k={BASELINE_K} omitted)")

    # Pack results
    all_binned = []

    for outcome in OUTCOMES:
        if outcome not in d.columns:
            print("\nSKIP (missing outcome column):", outcome)
            continue

        print("\n==============================")
        print("POINT EVENT STUDY:", outcome)

        tab, jt, diag = point_event_study(d, outcome)

        print("N obs:", diag["n_obs"], "countries:", diag["n_countries"], "treated countries in reg:", diag["treated_countries_in_reg"])
        print("Years:", diag["year_min"], "-", diag["year_max"])
        print("Controls used:", diag["controls"])
        print("\nPOINT effects (baseline k=-1 omitted):")
        print(tab[["k", "term", "coef", "se", "pval"]].to_string(index=False))

        if jt is None:
            print("\nJOINT PRETREND TEST (k=-5..-2): not available")
        else:
            fstat, pval, terms = jt
            print("\nJOINT PRETREND TEST (k=-5..-2):")
            print("Terms used:", terms)
            print("F-stat:", fstat, "p-value:", pval)

        point_out = f"eventstudy_point_results_{outcome}.csv"
        tab.to_csv(point_out, index=False)
        print("Saved:", point_out)

        # plot
        plot_path = plot_event_study(tab, outcome, pretrend_info=jt)
        print("Saved:", plot_path)

        # binned
        btab = binned_event_study(d, outcome)
        binned_out = f"eventstudy_binned_results_{outcome}.csv"
        btab.to_csv(binned_out, index=False)
        print("Saved:", binned_out)

        all_binned.append(btab)

    if all_binned:
        pack = pd.concat(all_binned, ignore_index=True)
        pack.to_csv("eventstudy_binned_results_ALL_outcomes.csv", index=False)
        print("\nSaved: eventstudy_binned_results_ALL_outcomes.csv")

    print("\nDONE.")


if __name__ == "__main__":
    main()


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# --- CONFIG ---
DATA = "../data/qarot_data/qarot_panel_min.csv"
OUTDIR = "../data/eventstudy_outputs_qarot_only"
COHORT_MIN, COHORT_MAX = 2002, 2010
W_PRE, W_POST = 5, 5
BASE_K = -1
# --------------

os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(DATA)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["treat_year"] = pd.to_numeric(df["treat_year"], errors="coerce").astype("Int64")
df["PARL"] = pd.to_numeric(df["PARL"], errors="coerce")

# cohort restriction: treated in [COHORT_MIN, COHORT_MAX] + never-treated
df = df[(df["treat_year"].isna()) | ((df["treat_year"] >= COHORT_MIN) & (df["treat_year"] <= COHORT_MAX))].copy()

cohort_years = list(range(COHORT_MIN, COHORT_MAX + 1))

# --- build stacked dataset ---
stacks = []
for g in cohort_years:
    treated_ctys = df.loc[df["treat_year"] == g, "Country_Code"].dropna().unique().tolist()
    if len(treated_ctys) == 0:
        continue

    # controls: never-treated OR treated later than g + w_post
    control_ctys = df.loc[df["treat_year"].isna() | (df["treat_year"] > (g + W_POST)), "Country_Code"].dropna().unique().tolist()

    dfg = df[df["Country_Code"].isin(treated_ctys + control_ctys)].copy()

    dfg["stack"] = int(g)
    dfg["rel_year"] = dfg["Year"].astype(int) - int(g)
    dfg = dfg[(dfg["rel_year"] >= -W_PRE) & (dfg["rel_year"] <= W_POST)].copy()
    dfg["treated_stack"] = dfg["Country_Code"].isin(treated_ctys).astype(int)
    stacks.append(dfg)

if not stacks:
    raise RuntimeError("Не удалось собрать ни одного стека (проверь COHORT_MIN/COHORT_MAX).")

d = pd.concat(stacks, ignore_index=True)
d = d.dropna(subset=["PARL"]).copy()

# Оставляем только те (stack, rel_year), где есть и treated, и control
mm = d.groupby(["stack", "rel_year"])["treated_stack"].agg(["min", "max"])
good_idx = mm[(mm["min"] == 0) & (mm["max"] == 1)].index
d = d.set_index(["stack", "rel_year"]).loc[good_idx].reset_index()


# FE ids
d["fe_unit"] = d["Country_Code"].astype(str) + "__" + d["stack"].astype(str)
d["fe_year"] = d["Year"].astype(str) + "__" + d["stack"].astype(str)

# event dummies
k_list = [k for k in range(-W_PRE, W_POST + 1) if k != BASE_K]
evt_terms = []
for k in k_list:
    term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
    d[term] = ((d["treated_stack"] == 1) & (d["rel_year"] == k)).astype(int)
    evt_terms.append(term)

lead_terms = [f"evt_{k:+d}".replace("+", "p").replace("-", "m") for k in range(-W_PRE, 0) if k != BASE_K and k <= -2]

print("STACKED SAMPLE (PARL)")
print("n_obs:", len(d), "countries:", d["Country_Code"].nunique(), "treated countries (in reg):", d.loc[d["treated_stack"]==1, "Country_Code"].nunique())
print("years:", int(d["Year"].min()), "-", int(d["Year"].max()))
print("cohorts used:", sorted(d["stack"].unique())[:5], "...", sorted(d["stack"].unique())[-5:])

from linearmodels.panel import PanelOLS
import numpy as np
import pandas as pd

# d уже построен у тебя выше, evt_terms уже есть

d2 = d.copy()

# linearmodels требует MultiIndex (entity, time)
d2["entity_id"] = d2["Country_Code"].astype(str) + "__" + d2["stack"].astype(int).astype(str)
d2["time_id"]   = d2["stack"].astype(int) * 10000 + d2["Year"].astype(int)

d2 = d2.set_index(["entity_id", "time_id"]).sort_index()

y = d2["PARL"]
X = d2[evt_terms].copy()

# выкинуть неидентифицируемые колонки (константные после фильтров)
X = X[[c for c in X.columns if X[c].nunique(dropna=False) > 1]]

# кластер по стране (а не по entity_id)
clusters = pd.Series(
    d2.index.get_level_values(0).str.split("__").str[0].values,
    index=d2.index
)

mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res = mod.fit(cov_type="clustered", clusters=clusters)

print(res.summary)

# Таблица эффектов
out = pd.DataFrame({
    "term": res.params.index,
    "coef": res.params.values,
    "se": res.std_errors.values,
    "pval": res.pvalues.values,
})

def term_to_k(t):
    if t.startswith("evt_m"):
        return -int(t.replace("evt_m",""))
    if t.startswith("evt_p"):
        return int(t.replace("evt_p",""))
    raise ValueError(t)

out["k"] = out["term"].apply(term_to_k)
out = out.sort_values("k")
print(out.to_string(index=False))

# Joint pretrend test на k=-5..-2
lead_terms = [t for t in out["term"] if term_to_k(t) <= -2]
param_names = list(res.params.index)

idx = [param_names.index(t) for t in lead_terms]
R = np.zeros((len(idx), len(param_names)))
for r,i in enumerate(idx):
    R[r,i] = 1.0

wt = res.wald_test(R)
pre_p = float(wt.pval)      # <-- добавь
pre_F = float(wt.stat)

plt.title(f"...\nJoint pretrend p={pre_p:.3f}")
# в linearmodels объект печатается нормально, но на всякий:
try:
    print("\nJOINT PRETREND TEST (k=-5..-2):")
    print("stat:", float(wt.stat), "p-value:", float(wt.pval))
except Exception:
    print("\nJOINT PRETREND TEST object:")
    print(wt)

# save csv
csv_path = os.path.join(OUTDIR, f"eventstudy_point_PARL_coh{COHORT_MIN}-{COHORT_MAX}_w{W_PRE}_{W_POST}.csv")
out.to_csv(csv_path, index=False)

# plot
plt.figure(figsize=(11,5))
plt.axhline(0, linewidth=1)
plt.axvline(BASE_K, linestyle="--", linewidth=1)
plt.errorbar(out["k"], out["coef"], yerr=1.96*out["se"], fmt="o", capsize=3)
plt.title(f"QAROT-only stacked event-study: PARL (baseline k={BASE_K} omitted)\nJoint pretrend p={pre_p:.3f}")
plt.xlabel("Event time k (years relative to quota implementation year)")
plt.ylabel("Effect on PARL (pp), vs baseline")
plt.tight_layout()
png_path = os.path.join(OUTDIR, f"eventstudy_plot_PARL_coh{COHORT_MIN}-{COHORT_MAX}_w{W_PRE}_{W_POST}.png")
plt.savefig(png_path, dpi=150)
plt.show()

print("\nSaved:")
print("CSV:", csv_path)
print("PNG:", png_path)


In [ ]:
!pip -q install linearmodels

import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS

# --- CONFIG ---
PANEL = "../paper4_parl_vs_hci/merged_panel_qarot_cohorts_1997_2022_window_m5_p12_v6.csv"  # <-- поменяй на свой merged panel
OUTDIR = "../data/eventstudy_outputs_panelols"
COHORT_MIN, COHORT_MAX = 2002, 2010
W_PRE, W_POST = 5, 12
BASE_K = -1

OUTCOMES = [
    "Human_Capital", "Life_Expectancy", "Female_Labor_Force",
    "Women_Business", "Women_Civil_Society", "Female_Adult_Literacy", "Fertility",
    # добавишь позже: "pwt_hc", "edu_index_z", ...
]

CONTROLS = ["GDP", "Political_Stability", "Corruption"]   # базовый набор; будет лагироваться
# --------------

os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(PANEL)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["treat_year"] = pd.to_numeric(df["treat_year"], errors="coerce").astype("Int64")

# cohort restriction
df = df[(df["treat_year"].isna()) | ((df["treat_year"]>=COHORT_MIN) & (df["treat_year"]<=COHORT_MAX))].copy()

# lag controls заранее (по стране и году)
df = df.sort_values(["Country_Code","Year"])
for c in CONTROLS:
    if c in df.columns:
        df[f"lag1_{c}"] = df.groupby("Country_Code")[c].shift(1)


In [ ]:
def build_stacked(df, cohort_years, w_pre, w_post):
    stacks = []
    for g in cohort_years:
        treated_ctys = df.loc[df["treat_year"]==g, "Country_Code"].dropna().unique().tolist()
        if not treated_ctys:
            continue

        control_ctys = df.loc[df["treat_year"].isna() | (df["treat_year"] > (g + w_post)), "Country_Code"] \
                        .dropna().unique().tolist()

        dfg = df[df["Country_Code"].isin(treated_ctys + control_ctys)].copy()
        dfg["stack"] = int(g)
        dfg["rel_year"] = dfg["Year"].astype(int) - int(g)
        dfg = dfg[(dfg["rel_year"]>=-w_pre) & (dfg["rel_year"]<=w_post)].copy()
        dfg["treated_stack"] = dfg["Country_Code"].isin(treated_ctys).astype(int)
        stacks.append(dfg)

    d = pd.concat(stacks, ignore_index=True)

    # оставляем только (stack, rel_year), где есть и treated и control
    mm = d.groupby(["stack","rel_year"])["treated_stack"].agg(["min","max"])
    good = mm[(mm["min"]==0) & (mm["max"]==1)].index
    d = d.set_index(["stack","rel_year"]).loc[good].reset_index()
    return d


def run_eventstudy_panelols(d, outcome, w_pre, w_post, base_k, controls):
    dd = d.dropna(subset=[outcome]).copy()

    # event dummies
    k_list = [k for k in range(-w_pre, w_post+1) if k!=base_k]
    evt_terms = []
    for k in k_list:
        term = f"evt_{k:+d}".replace("+","p").replace("-","m")
        dd[term] = ((dd["treated_stack"]==1) & (dd["rel_year"]==k)).astype(int)
        evt_terms.append(term)

    # X: evt + lag controls
    xcols = evt_terms[:]
    for c in controls:
        lc = f"lag1_{c}"
        if lc in dd.columns:
            xcols.append(lc)

    # PanelOLS index
    dd["entity_id"] = dd["Country_Code"].astype(str) + "__" + dd["stack"].astype(int).astype(str)
    dd["time_id"]   = dd["stack"].astype(int)*10000 + dd["Year"].astype(int)
    dd = dd.set_index(["entity_id","time_id"]).sort_index()

    y = dd[outcome]
    X = dd[xcols].copy()

    # убрать константные колонки
    X = X[[c for c in X.columns if X[c].nunique(dropna=False)>1]]

    clusters = pd.Series(
        dd.index.get_level_values(0).str.split("__").str[0].values,
        index=dd.index
    )

        # PanelOLS index
    dd["entity_id"] = dd["Country_Code"].astype(str) + "__" + dd["stack"].astype(int).astype(str)
    dd["time_id"]   = dd["stack"].astype(int)*10000 + dd["Year"].astype(int)
    dd = dd.set_index(["entity_id","time_id"]).sort_index()

    y = dd[outcome]
    X = dd[xcols].copy()

    # 1) drop missing jointly (важно!)
    data = pd.concat([y, X], axis=1).dropna()
    y = data[outcome]
    X = data.drop(columns=[outcome])

    # убрать константные колонки ещё раз (после dropna)
    X = X.loc[:, [c for c in X.columns if X[c].nunique(dropna=False) > 1]]

    # clusters (по исходной стране)
    clusters = pd.Series(
        data.index.get_level_values(0).str.split("__").str[0].values,
        index=data.index
    )

    # 2) drop_absorbed=True, 3) check_rank=False
    mod = PanelOLS(y, X, entity_effects=True, time_effects=True,
                   drop_absorbed=True, check_rank=False)
    res = mod.fit(cov_type="clustered", clusters=clusters)

    # таблица коэффициентов (только event terms)
    rows = []
    for term in evt_terms:
        if term in res.params.index:
            k = -int(term.replace("evt_m","")) if term.startswith("evt_m") else int(term.replace("evt_p",""))
            rows.append({
                "k": k,
                "term": term,
                "coef": float(res.params[term]),
                "se": float(res.std_errors[term]),
                "pval": float(res.pvalues[term]),
            })
    out = pd.DataFrame(rows).sort_values("k")

    # pretends: k<=-2
    lead_terms = [t for t in out["term"] if (t.startswith("evt_m") and int(t.replace("evt_m",""))>=2)]
    pre_p = np.nan
    if lead_terms:
        pnames = list(res.params.index)
        idx = [pnames.index(t) for t in lead_terms if t in pnames]
        R = np.zeros((len(idx), len(pnames)))
        for r,i in enumerate(idx): R[r,i] = 1.0
        wt = res.wald_test(R)
        pre_p = float(wt.pval)

    return res, out, pre_p


In [ ]:
cohort_years = list(range(COHORT_MIN, COHORT_MAX+1))
d = build_stacked(df, cohort_years, W_PRE, W_POST)

summary = []
for out in OUTCOMES:
    if out not in d.columns:
        continue

    res, tab, pre_p = run_eventstudy_panelols(d, out, W_PRE, W_POST, BASE_K, CONTROLS)

    # save csv
    tab_path = os.path.join(OUTDIR, f"eventstudy_point_{out}_w{W_PRE}_{W_POST}.csv")
    tab.to_csv(tab_path, index=False)

    # plot
    plt.figure(figsize=(11,5))
    plt.axhline(0, linewidth=1)
    plt.axvline(BASE_K, linestyle="--", linewidth=1)
    plt.errorbar(tab["k"], tab["coef"], yerr=1.96*tab["se"], fmt="o", capsize=3)
    plt.title(f"{out} (baseline k={BASE_K} omitted) | pretrend p={pre_p:.3f}")
    plt.xlabel("Event time k")
    plt.ylabel("Effect vs baseline")
    plt.tight_layout()
    png_path = os.path.join(OUTDIR, f"eventstudy_plot_{out}_w{W_PRE}_{W_POST}.png")
    plt.savefig(png_path, dpi=150)
    plt.close()

    summary.append({
        "outcome": out,
        "pretrend_p": pre_p,
        "avg_post_0_5": tab[(tab["k"]>=0) & (tab["k"]<=5)]["coef"].mean(),
        "n_obs": int(res.nobs),
    })

pd.DataFrame(summary).sort_values("pretrend_p").to_csv(os.path.join(OUTDIR,"summary.csv"), index=False)
print("Saved to:", OUTDIR)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS

# ============================================================
# CONFIG (правь только это)
# ============================================================

# 1) ТВОЙ ОСНОВНОЙ ДАТАСЕТ (где есть outcomes + контроли)
INPUT_CSV = "../data/qarot_data/qarot_data.csv"   # <-- замени на твой merged panel с GDP и т.п.

# 2) ИСТОЧНИК treat_year (если в INPUT_CSV его нет)
TREAT_SOURCE_CSV = "../data/qarot_data/qarot_panel_min.csv"  # <-- у тебя он уже есть

OUTDIR = "../data/eventstudy_outputs_panelols"

COHORT_MIN, COHORT_MAX = 2002, 2010
W_PRE, W_POST = 5, 12
BASE_K = -1

CONTROL_COLS = ["GDP", "Political_Stability", "Corruption"]  # лаг-1

OUTCOMES = [
    "PARL",
    "Human_Capital",
    "Life_Expectancy",
    "Female_Labor_Force",
    "Female_Adult_Literacy",
    "Women_Business",
    "Women_Civil_Society",
    "Fertility",
    # если есть:
    "edu_index_z",
    "health_index_z",
    "composite_hc_z",
]

# ============================================================
# Helpers
# ============================================================

def ensure_panel_keys(df: pd.DataFrame) -> pd.DataFrame:
    # Country_Code
    if "Country_Code" not in df.columns:
        if "code" in df.columns:
            df = df.rename(columns={"code": "Country_Code"})
        elif "Country Code" in df.columns:
            df = df.rename(columns={"Country Code": "Country_Code"})
    # Year
    if "Year" not in df.columns:
        if "year" in df.columns:
            df = df.rename(columns={"year": "Year"})
    # PARL (если QAROT raw)
    if "PARL" not in df.columns and "women.rep" in df.columns:
        df["PARL"] = df["women.rep"]

    if "Country_Code" not in df.columns or "Year" not in df.columns:
        raise RuntimeError(f"Не вижу ключи панели. Нужны Country_Code и Year. Колонки сейчас: {list(df.columns)[:40]} ...")

    df["Country_Code"] = df["Country_Code"].astype(str)
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

def attach_treat_year(df: pd.DataFrame, treat_source_csv: str) -> pd.DataFrame:
    if "treat_year" in df.columns:
        df["treat_year"] = pd.to_numeric(df["treat_year"], errors="coerce").astype("Int64")
        return df

    if not os.path.exists(treat_source_csv):
        raise RuntimeError("В INPUT_CSV нет treat_year и не найден TREAT_SOURCE_CSV. Укажи правильный путь к файлу с treat_year.")

    t = pd.read_csv(treat_source_csv)
    t = ensure_panel_keys(t)
    if "treat_year" not in t.columns:
        raise RuntimeError(f"В TREAT_SOURCE_CSV тоже нет treat_year. Колонки: {t.columns.tolist()}")

    t["treat_year"] = pd.to_numeric(t["treat_year"], errors="coerce").astype("Int64")
    t = t[["Country_Code", "treat_year"]].drop_duplicates("Country_Code")

    df = df.merge(t, on="Country_Code", how="left")
    if df["treat_year"].isna().all():
        raise RuntimeError("После merge treat_year всё равно пустой. Проверь совпадение ISO3 в Country_Code.")
    return df

def zscore_within_year(df: pd.DataFrame, col: str) -> pd.Series:
    def _z(s):
        m = s.mean(skipna=True)
        sd = s.std(skipna=True)
        if sd is None or sd == 0 or np.isnan(sd):
            return pd.Series([np.nan] * len(s), index=s.index)
        return (s - m) / sd
    return df.groupby("Year")[col].transform(_z)

def build_indices_if_possible(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for c in ["Life_Expectancy", "SP.DYN.IMRT.IN", "SE.PRM.ENRR.FE", "SE.SEC.ENRR.FE", "SE.TER.ENRR.FE", "SE.ADT.LITR.FE.ZS", "Female_Adult_Literacy"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    edu_parts = [c for c in ["SE.PRM.ENRR.FE", "SE.SEC.ENRR.FE", "SE.TER.ENRR.FE", "SE.ADT.LITR.FE.ZS", "Female_Adult_Literacy"] if c in out.columns]
    if edu_parts and "edu_index_z" not in out.columns:
        out["edu_index_z"] = pd.concat([zscore_within_year(out, c) for c in edu_parts], axis=1).mean(axis=1, skipna=True)

    if ("Life_Expectancy" in out.columns) and ("SP.DYN.IMRT.IN" in out.columns) and ("health_index_z" not in out.columns):
        out["health_index_z"] = zscore_within_year(out, "Life_Expectancy") - zscore_within_year(out, "SP.DYN.IMRT.IN")

    if ("composite_hc_z" not in out.columns) and (("edu_index_z" in out.columns) or ("health_index_z" in out.columns)):
        parts = []
        if "edu_index_z" in out.columns: parts.append(out["edu_index_z"])
        if "health_index_z" in out.columns: parts.append(out["health_index_z"])
        out["composite_hc_z"] = pd.concat(parts, axis=1).mean(axis=1, skipna=True)

    return out

def make_stacked_sample(df: pd.DataFrame, cohort_years, w_pre, w_post) -> pd.DataFrame:
    stacks = []
    for g in cohort_years:
        treated_ctys = df.loc[df["treat_year"] == g, "Country_Code"].dropna().unique().tolist()
        if not treated_ctys:
            continue

        control_ctys = df.loc[
            df["treat_year"].isna() | (df["treat_year"] > (g + w_post)),
            "Country_Code"
        ].dropna().unique().tolist()

        dfg = df[df["Country_Code"].isin(treated_ctys + control_ctys)].copy()
        dfg["stack"] = int(g)
        dfg["rel_year"] = dfg["Year"].astype(int) - int(g)
        dfg = dfg[(dfg["rel_year"] >= -w_pre) & (dfg["rel_year"] <= w_post)].copy()
        dfg["treated_stack"] = dfg["Country_Code"].isin(treated_ctys).astype(int)
        stacks.append(dfg)

    if not stacks:
        raise RuntimeError("No stacks created. Проверь COHORT_MIN/COHORT_MAX и наличие treat_year.")

    d = pd.concat(stacks, ignore_index=True)

    # Оставляем только те (stack, rel_year), где есть и treated, и control
    mm = d.groupby(["stack", "rel_year"])["treated_stack"].agg(["min", "max"])
    good = mm[(mm["min"] == 0) & (mm["max"] == 1)].index
    d = d.set_index(["stack", "rel_year"]).loc[good].reset_index()

    # FE ids
    d["fe_unit"] = d["Country_Code"].astype(str) + "__" + d["stack"].astype(str)  # entity = country×stack

    # ВАЖНО: time index должен быть numeric/date-like
    d["time_id"] = d["stack"].astype(int) * 10000 + d["Year"].astype(int)         # time = (stack,Year) numeric id

    return d

def save_plot(tab: pd.DataFrame, title: str, out_png: str, base_k: int):
    plt.figure(figsize=(11, 5))
    plt.axhline(0, linewidth=1)
    plt.axvline(base_k, linestyle="--", linewidth=1)
    plt.errorbar(tab["k"], tab["coef"], yerr=1.96 * tab["se"], fmt="o", capsize=3)
    plt.title(title)
    plt.xlabel("Event time k")
    plt.ylabel("Coef vs baseline")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def run_eventstudy_panelols(d: pd.DataFrame, outcome: str, w_pre: int, w_post: int, base_k: int, controls: list):
    dd = d.copy()
    if outcome not in dd.columns:
        return None, None, np.nan

    dd[outcome] = pd.to_numeric(dd[outcome], errors="coerce")
    dd = dd.dropna(subset=[outcome, "Year", "time_id", "fe_unit"]).copy()

    # event dummies
    k_list = [k for k in range(-w_pre, w_post + 1) if k != base_k]
    evt_terms = []
    for k in k_list:
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        dd[term] = ((dd["treated_stack"] == 1) & (dd["rel_year"] == k)).astype(int)
        evt_terms.append(term)

    # lag controls
    dd = dd.sort_values(["Country_Code", "Year"])
    lag_terms = []
    for c in controls:
        if c in dd.columns:
            dd[c] = pd.to_numeric(dd[c], errors="coerce")
            dd[f"lag1_{c}"] = dd.groupby("Country_Code")[c].shift(1)
            lag_terms.append(f"lag1_{c}")

    exog_cols = evt_terms + lag_terms
    X = dd[exog_cols].copy()

    # drop missing exog
    keep = X.notna().all(axis=1)
    dd = dd.loc[keep].copy()
    X = X.loc[keep].copy()

    # drop constant/empty columns
    X = X.loc[:, X.notna().any(axis=0)]
    X = X.loc[:, (X.nunique(dropna=False) > 1)]
    X = X.loc[:, (X.fillna(0).abs().sum(axis=0) > 0)]

    if X.shape[1] == 0:
        raise RuntimeError(f"No usable regressors left for {outcome} (absorbed/constant).")

    dd = dd.set_index(["fe_unit", "time_id"])
    X = X.set_index(dd.index)
    y = dd[outcome]

    clusters = pd.Series(
    dd.index.get_level_values(0).astype(str).str.split("__").str[0].to_numpy(),
    index=dd.index
    )


    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = PanelOLS(
            y, X,
            entity_effects=True,
            time_effects=True,      # FE for (stack,Year)
            drop_absorbed=True,
            check_rank=False
        ).fit(cov_type="clustered", clusters=clusters)

    rows = []
    for k in range(-w_pre, w_post + 1):
        if k == base_k:
            continue
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        rows.append({
            "k": k,
            "term": term,
            "coef": float(res.params.get(term, np.nan)),
            "se": float(res.std_errors.get(term, np.nan)),
            "pval": float(res.pvalues.get(term, np.nan)),
        })
    tab = pd.DataFrame(rows).sort_values("k")

    # joint pretrend test: k=-w_pre..-2
    lead_terms = [f"evt_{k:+d}".replace("+", "p").replace("-", "m")
                  for k in range(-w_pre, 0) if (k != base_k and k <= -2)]
    present = [t for t in lead_terms if t in res.params.index]
    pre_p = np.nan
    try:
        if present:
            R = np.zeros((len(present), len(res.params)))
            pindex = list(res.params.index)
            for i, t in enumerate(present):
                R[i, pindex.index(t)] = 1.0
            w = res.wald_test(R, np.zeros(len(present)))
            pre_p = float(w.pval)
    except Exception:
        pre_p = np.nan

    return res, tab, pre_p

# ============================================================
# Main
# ============================================================

def main():
    os.makedirs(OUTDIR, exist_ok=True)

    df = pd.read_csv(INPUT_CSV)
    df = ensure_panel_keys(df)
    df = attach_treat_year(df, TREAT_SOURCE_CSV)
    df = build_indices_if_possible(df)

    # cohort restriction
    df = df[(df["treat_year"].isna()) | ((df["treat_year"] >= COHORT_MIN) & (df["treat_year"] <= COHORT_MAX))].copy()

    cohort_years = list(range(COHORT_MIN, COHORT_MAX + 1))
    d = make_stacked_sample(df, cohort_years, W_PRE, W_POST)

    print("STACKED BASE SAMPLE")
    print("n_obs:", len(d),
          "countries:", d["Country_Code"].nunique(),
          "treated countries (in stacks):", d.loc[d["treated_stack"] == 1, "Country_Code"].nunique())
    print("years:", int(d["Year"].min()), "-", int(d["Year"].max()))
    print("cohorts used:", sorted(d["stack"].unique())[:5], "...", sorted(d["stack"].unique())[-5:])

    outcomes = [o for o in OUTCOMES if o in d.columns]
    if not outcomes:
        raise RuntimeError(f"Ни один outcome из OUTCOMES не найден. Доступные колонки: {list(d.columns)[:60]} ...")

    summaries = []
    for out in outcomes:
        controls = [c for c in CONTROL_COLS if (c in d.columns and c != out)]
        if out == "Fertility":
            controls = [c for c in controls if c != "Fertility"]

        try:
            res, tab, pre_p = run_eventstudy_panelols(d, out, W_PRE, W_POST, BASE_K, controls)
        except Exception as e:
            print(f"[FAIL] {out}: {e}")
            continue

        tab_path = os.path.join(OUTDIR, f"eventstudy_point_{out}_w{W_PRE}_{W_POST}.csv")
        tab.to_csv(tab_path, index=False)

        png_path = os.path.join(OUTDIR, f"eventstudy_plot_{out}_w{W_PRE}_{W_POST}.png")
        title = f"Stacked ES (PanelOLS): {out} | cohorts {COHORT_MIN}-{COHORT_MAX} | window [-{W_PRE},+{W_POST}] | pretrend p={pre_p:.3f}"
        save_plot(tab, title, png_path, BASE_K)

        avg_post_0_5 = tab[(tab["k"] >= 0) & (tab["k"] <= 5)]["coef"].mean()
        avg_lead = tab[(tab["k"] <= -2) & (tab["k"] >= -W_PRE)]["coef"].mean()

        summaries.append({
            "outcome": out,
            "n_obs_used": int(res.nobs),
            "controls_lag1": ",".join([f"lag1_{c}" for c in controls]) if controls else "",
            "pretrend_p": pre_p,
            "avg_lead_-pre..-2": avg_lead,
            "avg_post_0..5": avg_post_0_5,
            "csv": tab_path,
            "png": png_path,
        })

        print(f"[OK] {out}: pretrend_p={pre_p:.4f} avg_post_0..5={avg_post_0_5:.4f} -> {tab_path}")

    if not summaries:
        print("\nНичего не оценилось. Смотри [FAIL] выше.")
        return

    summ = pd.DataFrame(summaries).sort_values("outcome")
    summ_path = os.path.join(OUTDIR, "summary.csv")
    summ.to_csv(summ_path, index=False)
    print("\nSaved summary to:", summ_path)
    print("All outputs in:", OUTDIR)

main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Stacked event-study (quota adoption -> outcomes) using PanelOLS (linearmodels).

WHAT IT DOES
1) Loads your main panel (final_data.csv) and QAROT (minimal panel with treat_year + PARL).
2) Merges treat_year + PARL into the main panel.
3) Builds optional simple HC proxies (edu_index_z, health_index_z, composite_hc_z) from what you already have.
4) Builds stacked sample (Sun & Abraham-style stacking by cohort).
5) Runs PanelOLS with FE:
   - entity FE = (country x stack)
   - time FE   = (year x stack)  [implemented via numeric time_id]
   Clustered SE by country.
6) Saves per-outcome CSV + PNG, and a summary.csv.

DEPENDENCIES
pip install pandas numpy matplotlib linearmodels

USAGE
- In Jupyter: paste and run (it will run main()).
- As a script: python run_stacked_es.py
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# linearmodels is required
try:
    from linearmodels.panel import PanelOLS
except Exception as e:
    raise RuntimeError("Install linearmodels: pip install linearmodels") from e


# =========================
# CONFIG (EDIT THESE)
# =========================
MAIN_PANEL_CSV = "../data/final_data/final_data.csv"          # your main panel (semicolon + comma decimals in your case)
QAROT_MIN_CSV  = "../data/qarot_data/qarot_panel_min.csv"     # must contain: Country_Code, Year, treat_year, PARL
# OPTIONAL fallback if qarot_panel_min.csv doesn't exist:
QAROT_RAW_CSV  = "../data/qarot_data/qarot_data.csv"          # raw QAROT (if you want the script to rebuild minimal)

OUTDIR         = "../data/eventstudy_outputs_panelols"

COHORT_MIN, COHORT_MAX = 2002, 2010
W_PRE, W_POST          = 5, 12
BASE_K                 = -1

# controls (will be lagged by 1 year; only those found in data are used)
CONTROL_COLS = ["GDP", "Corruption", "Political_Stability", "Government_Eff", "Rule_of_Law", "Regulatory_Quality"]

# outcomes to estimate (only those present in merged data will be used)
OUTCOMES = [
    "PARL",
    "Human_Capital",
    "Life_Expectancy",
    "Female_Labor_Force",
    "Female_Adult_Literacy",
    "Fertility",
    "Women_Business",
    "Women_Civil_Society",
    # optional constructed proxies
    "edu_index_z",
    "health_index_z",
    "composite_hc_z",
]
# =========================


# -------------------------
# Helpers
# -------------------------
def read_csv_smart(path: str) -> tuple[pd.DataFrame, dict]:
    """
    Tries to read CSV with common settings.
    Returns (df, used_kwargs).
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")

    attempts = [
        {"sep": ",", "engine": "python"},
        {"sep": ";", "engine": "python", "decimal": ","},
        {"sep": ";", "engine": "python"},
        {"sep": ",", "engine": "python", "decimal": ","},
    ]
    last_err = None
    for kw in attempts:
        try:
            df = pd.read_csv(path, **kw)
            return df, kw
        except Exception as e:
            last_err = e
            continue
    raise last_err


def zscore(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    m = s.mean(skipna=True)
    sd = s.std(skipna=True)
    if sd is None or sd == 0 or np.isnan(sd):
        return pd.Series(np.nan, index=s.index)
    return (s - m) / sd


def rebuild_qarot_min_if_needed() -> None:
    """
    If QAROT_MIN_CSV is missing, try to build it from QAROT_RAW_CSV.
    Expected raw columns (your case): code, year, implemented.quota, women.rep
    """
    if os.path.exists(QAROT_MIN_CSV):
        return

    if not os.path.exists(QAROT_RAW_CSV):
        raise FileNotFoundError(
            f"QAROT minimal panel missing: {QAROT_MIN_CSV}\n"
            f"And raw QAROT also missing: {QAROT_RAW_CSV}\n"
            f"Either create {QAROT_MIN_CSV} first or point QAROT_RAW_CSV to your raw file."
        )

    dfq, kw = read_csv_smart(QAROT_RAW_CSV)

    # normalize names
    low = {c: c.lower() for c in dfq.columns}
    dfq = dfq.rename(columns=low)

    # country iso3
    if "code" not in dfq.columns:
        raise RuntimeError("QAROT raw: expected column 'code' (ISO3).")
    # year
    if "year" not in dfq.columns:
        raise RuntimeError("QAROT raw: expected column 'year'.")
    # treat year source
    if "implemented.quota" not in dfq.columns and "adopted.quota" not in dfq.columns:
        raise RuntimeError("QAROT raw: expected 'implemented.quota' or 'adopted.quota'.")

    treat_src = "implemented.quota" if "implemented.quota" in dfq.columns else "adopted.quota"
    parl_src = "women.rep" if "women.rep" in dfq.columns else None
    if parl_src is None:
        raise RuntimeError("QAROT raw: expected 'women.rep' for PARL.")

    out = dfq[["code", "year", treat_src, parl_src]].copy()
    out = out.rename(columns={"code": "Country_Code", "year": "Year", treat_src: "treat_year", parl_src: "PARL"})

    out["Year"] = pd.to_numeric(out["Year"], errors="coerce").astype("Int64")
    out["treat_year"] = pd.to_numeric(out["treat_year"], errors="coerce").astype("Int64")
    out["PARL"] = pd.to_numeric(out["PARL"], errors="coerce")

    # propagate treat_year to be constant per country (first non-null)
    ty = out.groupby("Country_Code")["treat_year"].transform(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA)
    out["treat_year"] = ty.astype("Int64")

    os.makedirs(os.path.dirname(QAROT_MIN_CSV), exist_ok=True)
    out.to_csv(QAROT_MIN_CSV, index=False)
    print(f"[QAROT MIN BUILT] Saved: {QAROT_MIN_CSV} | shape={out.shape}")


def load_and_merge() -> pd.DataFrame:
    rebuild_qarot_min_if_needed()

    df_main, kw_main = read_csv_smart(MAIN_PANEL_CSV)
    df_q, kw_q = read_csv_smart(QAROT_MIN_CSV)

    print(f"[read_csv OK] final_data.csv using {kw_main} shape={df_main.shape}")
    print(f"[read_csv OK] qarot_panel_min.csv using {kw_q} shape={df_q.shape}")

    # normalize main keys
    if "Country_Code" not in df_main.columns:
        raise RuntimeError("MAIN_PANEL_CSV must contain Country_Code.")
    if "Year" not in df_main.columns:
        # common fallback
        if "year" in df_main.columns:
            df_main = df_main.rename(columns={"year": "Year"})
        else:
            raise RuntimeError("MAIN_PANEL_CSV must contain Year (or year).")

    df_main["Year"] = pd.to_numeric(df_main["Year"], errors="coerce").astype("Int64")

    # normalize qarot keys
    for c in ["Country_Code", "Year", "treat_year", "PARL"]:
        if c not in df_q.columns:
            raise RuntimeError(f"QAROT_MIN_CSV must contain column: {c}")

    df_q["Year"] = pd.to_numeric(df_q["Year"], errors="coerce").astype("Int64")
    df_q["treat_year"] = pd.to_numeric(df_q["treat_year"], errors="coerce").astype("Int64")
    df_q["PARL"] = pd.to_numeric(df_q["PARL"], errors="coerce")

    # merge treat_year + PARL into main (left join)
    df = df_main.merge(df_q[["Country_Code", "Year", "treat_year", "PARL"]], on=["Country_Code", "Year"], how="left")

    # treat_year should be constant per country -> propagate within country after merge
    df["treat_year"] = df.groupby("Country_Code")["treat_year"].transform(
        lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA
    ).astype("Int64")

    # also propagate PARL (QAROT-only series) within country-year already aligned; keep as is (panel may have gaps)
    return df


def build_optional_indices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build simple proxies from columns you already have (no downloads):
    - edu_index_z: z(Female_Adult_Literacy) (if exists)
    - health_index_z: z(Life_Expectancy) - z(Fertility) (if both exist)
    - composite_hc_z: mean of available {z(Human_Capital), edu_index_z, health_index_z}
    """
    if "Female_Adult_Literacy" in df.columns:
        df["edu_index_z"] = zscore(df["Female_Adult_Literacy"])
    else:
        df["edu_index_z"] = np.nan

    if "Life_Expectancy" in df.columns and "Fertility" in df.columns:
        df["health_index_z"] = zscore(df["Life_Expectancy"]) - zscore(df["Fertility"])
    else:
        df["health_index_z"] = np.nan

    comps = []
    if "Human_Capital" in df.columns:
        comps.append(zscore(df["Human_Capital"]))
    if "edu_index_z" in df.columns:
        comps.append(df["edu_index_z"])
    if "health_index_z" in df.columns:
        comps.append(df["health_index_z"])

    df["composite_hc_z"] = pd.concat(comps, axis=1).mean(axis=1, skipna=True) if comps else np.nan
    return df


def add_lag_controls(df: pd.DataFrame, controls: list[str]) -> pd.DataFrame:
    df = df.sort_values(["Country_Code", "Year"]).copy()
    for c in controls:
        if c not in df.columns:
            continue
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df[f"lag1_{c}"] = df.groupby("Country_Code")[c].shift(1)
    return df


def build_stacked_sample(df: pd.DataFrame) -> pd.DataFrame:
    # cohort restriction: treated in range + never-treated
    df = df[(df["treat_year"].isna()) | ((df["treat_year"] >= COHORT_MIN) & (df["treat_year"] <= COHORT_MAX))].copy()

    cohorts = list(range(COHORT_MIN, COHORT_MAX + 1))
    stacks = []

    for g in cohorts:
        treated_ctys = df.loc[df["treat_year"] == g, "Country_Code"].dropna().unique().tolist()
        if len(treated_ctys) == 0:
            continue

        # controls: never-treated OR treated after g + W_POST (not yet treated within window)
        control_ctys = df.loc[df["treat_year"].isna() | (df["treat_year"] > (g + W_POST)), "Country_Code"].dropna().unique().tolist()

        dfg = df[df["Country_Code"].isin(treated_ctys + control_ctys)].copy()
        dfg["stack"] = int(g)
        dfg["rel_year"] = dfg["Year"].astype(int) - int(g)
        dfg = dfg[(dfg["rel_year"] >= -W_PRE) & (dfg["rel_year"] <= W_POST)].copy()
        dfg["treated_stack"] = dfg["Country_Code"].isin(treated_ctys).astype(int)
        stacks.append(dfg)

    if not stacks:
        raise RuntimeError("No stacks created. Check COHORT_MIN/COHORT_MAX vs treat_year coverage.")

    d = pd.concat(stacks, ignore_index=True)

    # keep only (stack, rel_year) cells with both treated and control
    mm = d.groupby(["stack", "rel_year"])["treated_stack"].agg(["min", "max"])
    good = mm[(mm["min"] == 0) & (mm["max"] == 1)].index
    d = d.set_index(["stack", "rel_year"]).loc[good].reset_index()

    # FE identifiers
    d["fe_unit"] = d["Country_Code"].astype(str) + "__" + d["stack"].astype(str)
    d["fe_time_key"] = d["Year"].astype(int).astype(str) + "__" + d["stack"].astype(str)

    # IMPORTANT: PanelOLS requires numeric/date-like time index -> encode fe_time_key to int codes
    d["time_id"] = d["fe_time_key"].astype("category").cat.codes.astype(int)

    print("STACKED BASE SAMPLE")
    print("n_obs:", len(d), "countries:", d["Country_Code"].nunique(), "treated countries (in stacks):", d.loc[d["treated_stack"] == 1, "Country_Code"].nunique())
    print("years:", int(d["Year"].min()), "-", int(d["Year"].max()))
    uu = sorted(d["stack"].unique().tolist())
    print("cohorts used:", uu[:5], "...", uu[-5:])

    return d


def make_event_terms(d: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    k_list = [k for k in range(-W_PRE, W_POST + 1) if k != BASE_K]
    evt_terms = []
    for k in k_list:
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        d[term] = ((d["treated_stack"] == 1) & (d["rel_year"] == k)).astype(int)
        evt_terms.append(term)

    lead_terms = [f"evt_{k:+d}".replace("+", "p").replace("-", "m") for k in range(-W_PRE, 0) if k != BASE_K and k <= -2]
    return d, evt_terms, lead_terms


def run_eventstudy_panelols(d: pd.DataFrame, outcome: str, evt_terms: list[str], lead_terms: list[str], lag_controls: list[str]):
    # drop missing outcome
    dd = d.dropna(subset=[outcome]).copy()

    # build panel index
    dd = dd.set_index(["fe_unit", "time_id"]).sort_index()

    y = pd.to_numeric(dd[outcome], errors="coerce")
    X_cols = evt_terms + lag_controls
    X = dd[X_cols].copy()

    # drop cols with all nan or constant
    keep = []
    for c in X.columns:
        s = X[c]
        if s.notna().sum() == 0:
            continue
        if s.nunique(dropna=False) <= 1:
            continue
        keep.append(c)
    X = X[keep]

    # align and drop rows with NA in y or X
    tmp = pd.concat([y, X], axis=1).dropna()
    y2 = tmp.iloc[:, 0]
    X2 = tmp.iloc[:, 1:]

    # clusters by country (strip "__stack" from entity index)
    ent = tmp.index.get_level_values(0).astype(str)
    cl = ent.str.split("__").str[0]
    clusters = pd.DataFrame({"country_cluster": cl.values}, index=tmp.index)

    # fit
    mod = PanelOLS(
        y2,
        X2,
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True,
        check_rank=False,
    )
    res = mod.fit(cov_type="clustered", clusters=clusters)

    # coef table for event terms (some may be absorbed/dropped)
    rows = []
    for k in range(-W_PRE, W_POST + 1):
        if k == BASE_K:
            continue
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        rows.append({
            "k": k,
            "term": term,
            "coef": float(res.params.get(term, np.nan)),
            "se": float(res.std_errors.get(term, np.nan)),
            "pval": float(res.pvalues.get(term, np.nan)),
        })
    tab = pd.DataFrame(rows).sort_values("k")

    # joint pretrend test on leads
    pre_p = np.nan
    try:
        params = res.params.index.tolist()
        leads = [t for t in lead_terms if t in params]
        if len(leads) > 0:
            R = np.zeros((len(leads), len(params)))
            for i, t in enumerate(leads):
                R[i, params.index(t)] = 1.0
            w = res.wald_test(R)  # works in linearmodels
            # w has pval in recent versions; otherwise fallback
            pre_p = float(getattr(w, "pval", getattr(w, "pvalue", np.nan)))
    except Exception:
        pre_p = np.nan

    return res, tab, pre_p


def plot_eventstudy(tab: pd.DataFrame, title: str, out_png: str, pre_p: float):
    plt.figure(figsize=(11, 5))
    plt.axhline(0, linewidth=1)
    plt.axvline(BASE_K, linestyle="--", linewidth=1)
    plt.errorbar(tab["k"], tab["coef"], yerr=1.96 * tab["se"], fmt="o", capsize=3)
    plt.title(f"{title}\npretrend p={pre_p:.3f}" if np.isfinite(pre_p) else title)
    plt.xlabel("Event time k")
    plt.ylabel("Coef vs baseline")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def main():
    os.makedirs(OUTDIR, exist_ok=True)

    df = load_and_merge()
    df = build_optional_indices(df)
    df = add_lag_controls(df, CONTROL_COLS)

    # build stacked base once (then reuse for all outcomes)
    d = build_stacked_sample(df)
    d, evt_terms, lead_terms = make_event_terms(d)

    # available lag controls
    lag_controls = [f"lag1_{c}" for c in CONTROL_COLS if f"lag1_{c}" in d.columns]

    summaries = []
    for out in OUTCOMES:
        if out not in d.columns:
            continue
        try:
            res, tab, pre_p = run_eventstudy_panelols(d, out, evt_terms, lead_terms, lag_controls)

            tab_path = os.path.join(OUTDIR, f"eventstudy_point_{out}_w{W_PRE}_{W_POST}.csv")
            tab.to_csv(tab_path, index=False)

            png_path = os.path.join(OUTDIR, f"eventstudy_plot_{out}_w{W_PRE}_{W_POST}.png")
            plot_eventstudy(tab, f"Stacked ES (PanelOLS): {out} | cohorts {COHORT_MIN}-{COHORT_MAX} | window (-{W_PRE},+{W_POST})", png_path, pre_p)

            avg_post_0_5 = float(tab[(tab["k"] >= 0) & (tab["k"] <= 5)]["coef"].mean())
            summaries.append({
                "outcome": out,
                "nobs": int(res.nobs),
                "pretrend_p": pre_p,
                "avg_post_0..5": avg_post_0_5,
                "csv": tab_path,
                "png": png_path,
            })
            print(f"[OK] {out}: pretrend_p={pre_p:.4f} avg_post_0..5={avg_post_0_5:.4f} -> {tab_path}")
        except Exception as e:
            print(f"[FAIL] {out}: {e}")

    if len(summaries) > 0:
        summ = pd.DataFrame(summaries).sort_values("outcome")
        summ_path = os.path.join(OUTDIR, "summary.csv")
        summ.to_csv(summ_path, index=False)
        print("\nSaved summary to:", summ_path)
    else:
        print("\nNo outcomes estimated (all failed or missing).")

    print("All outputs in:", OUTDIR)


if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
"""
ONE-CELL SCRIPT (Jupyter-friendly): build QAROT minimal panel (if needed),
merge with your main panel (final_data.csv), run STACKED event-study via PanelOLS,
save point-estimates CSVs + plots + summary.csv.

Required:
  pip install pandas numpy matplotlib linearmodels openpyxl

Files expected (adjust paths below if needed):
  - ../data/final_data/final_data.csv
  - ../data/qarot_data/qarot_data.csv   (raw QAROT)
    OR ../data/qarot_data/qarot_panel_min.csv (already built)
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS

# =========================
# CONFIG (EDIT HERE)
# =========================
MAIN_PANEL_CSV = "../data/final_data/final_data.csv"

QAROT_RAW_CSV  = "../data/qarot_data/qarot_data.csv"
QAROT_MIN_CSV  = "../data/qarot_data/qarot_panel_min.csv"   # will be created if missing

MERGED_DIR     = "../data/merged_panel"
MERGED_CSV     = os.path.join(MERGED_DIR, "merged_panel.csv")

OUTDIR         = "../data/eventstudy_outputs_panelols"

COHORT_MIN, COHORT_MAX = 2002, 2010
W_PRE, W_POST          = 5, 12
BASE_K                 = -1

# Controls will be LAGGED by 1 year within (Country_Code, stack)
CONTROL_COLS = ["GDP", "Political_Stability", "Corruption", "Fertility"]
# =========================


# -------------------------
# Utilities
# -------------------------
def zscore(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    m = s.mean(skipna=True)
    sd = s.std(skipna=True)
    if sd is None or sd == 0 or np.isnan(sd):
        return pd.Series(np.nan, index=s.index)
    return (s - m) / sd

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def read_main_panel(path: str) -> pd.DataFrame:
    # your final_data.csv is ';' with decimal ','
    return pd.read_csv(path, sep=";", engine="python", decimal=",")

def normalize_country_year(df: pd.DataFrame) -> pd.DataFrame:
    # Standardize key names
    if "Country_Code" not in df.columns and "countrycode" in df.columns:
        df = df.rename(columns={"countrycode": "Country_Code"})
    if "Year" not in df.columns and "year" in df.columns:
        df = df.rename(columns={"year": "Year"})
    if "Country_Code" not in df.columns or "Year" not in df.columns:
        raise RuntimeError("Need Country_Code and Year columns after normalization.")
    df["Country_Code"] = df["Country_Code"].astype(str)
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

def build_qarot_min_panel_if_needed() -> pd.DataFrame:
    """
    Produce minimal QAROT panel with:
      Country_Code, Year, treat_year, PARL
    From raw QAROT (common columns: code, year, women.rep, implemented.quota / first.implement.flag).
    """
    if os.path.exists(QAROT_MIN_CSV):
        dfq = pd.read_csv(QAROT_MIN_CSV, engine="python")
        dfq = normalize_country_year(dfq)
        if "treat_year" not in dfq.columns or "PARL" not in dfq.columns:
            raise RuntimeError("QAROT_MIN_CSV exists but must contain treat_year and PARL.")
        dfq["treat_year"] = pd.to_numeric(dfq["treat_year"], errors="coerce").astype("Int64")
        dfq["PARL"] = pd.to_numeric(dfq["PARL"], errors="coerce")
        return dfq[["Country_Code", "Year", "treat_year", "PARL"]].copy()

    if not os.path.exists(QAROT_RAW_CSV):
        raise FileNotFoundError(f"Neither {QAROT_MIN_CSV} nor {QAROT_RAW_CSV} exists.")

    df = pd.read_csv(QAROT_RAW_CSV, engine="python")
    # expected in raw: code (ISO3), year, women.rep
    if "code" not in df.columns or "year" not in df.columns:
        raise RuntimeError("Raw QAROT must contain columns: code, year (at least).")

    df = df.rename(columns={"code": "Country_Code", "year": "Year"})
    df = normalize_country_year(df)

    # PARL from women.rep if present
    if "women.rep" in df.columns and "PARL" not in df.columns:
        df["PARL"] = pd.to_numeric(df["women.rep"], errors="coerce")
    elif "PARL" in df.columns:
        df["PARL"] = pd.to_numeric(df["PARL"], errors="coerce")
    else:
        raise RuntimeError("Raw QAROT must contain women.rep (to build PARL) or PARL.")

    # treat_year: pick best available indicator
    treat_year = None
    if "first.implement.flag" in df.columns:
        tmp = df.copy()
        tmp["flag"] = pd.to_numeric(tmp["first.implement.flag"], errors="coerce")
        treat_year = (
            tmp.loc[tmp["flag"] == 1]
               .groupby("Country_Code")["Year"]
               .min()
        )
    elif "implemented.quota" in df.columns:
        tmp = df.copy()
        tmp["flag"] = pd.to_numeric(tmp["implemented.quota"], errors="coerce")
        treat_year = (
            tmp.loc[tmp["flag"] == 1]
               .groupby("Country_Code")["Year"]
               .min()
        )
    elif "implemented_reform_flag" in df.columns:
        tmp = df.copy()
        tmp["flag"] = pd.to_numeric(tmp["implemented_reform_flag"], errors="coerce")
        treat_year = (
            tmp.loc[tmp["flag"] == 1]
               .groupby("Country_Code")["Year"]
               .min()
        )
    else:
        raise RuntimeError("Raw QAROT must contain first.implement.flag or implemented.quota to build treat_year.")

    dfq = df[["Country_Code", "Year", "PARL"]].copy()
    dfq["treat_year"] = dfq["Country_Code"].map(treat_year).astype("Int64")

    # Save
    ensure_dir(os.path.dirname(QAROT_MIN_CSV))
    dfq_out = dfq[["Country_Code", "Year", "treat_year", "PARL"]].copy()
    dfq_out.to_csv(QAROT_MIN_CSV, index=False)
    print("Saved QAROT min panel:", QAROT_MIN_CSV)
    return dfq_out

def build_optional_indices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Indices that don't require external downloads:
      - edu_index_z: z(Female_Adult_Literacy) (if exists)
      - health_index_z: z(Life_Expectancy) (if exists)
      - composite_hc_z: mean of available z-scores among:
            z(Human_Capital), edu_index_z, health_index_z
    """
    if "Female_Adult_Literacy" in df.columns:
        df["edu_index_z"] = zscore(df["Female_Adult_Literacy"])
    else:
        df["edu_index_z"] = np.nan

    if "Life_Expectancy" in df.columns:
        df["health_index_z"] = zscore(df["Life_Expectancy"])
    else:
        df["health_index_z"] = np.nan

    hc_parts = []
    if "Human_Capital" in df.columns:
        df["human_capital_z"] = zscore(df["Human_Capital"])
        hc_parts.append(df["human_capital_z"])
    if "edu_index_z" in df.columns:
        hc_parts.append(df["edu_index_z"])
    if "health_index_z" in df.columns:
        hc_parts.append(df["health_index_z"])

    if hc_parts:
        df["composite_hc_z"] = pd.concat(hc_parts, axis=1).mean(axis=1, skipna=True)
    else:
        df["composite_hc_z"] = np.nan

    return df

def add_lag_controls(d: pd.DataFrame, controls: list) -> pd.DataFrame:
    # lag within (Country_Code, stack)
    d = d.sort_values(["Country_Code", "stack", "Year"]).copy()
    for c in controls:
        if c not in d.columns:
            d[c] = np.nan
        d[f"lag1_{c}"] = d.groupby(["Country_Code", "stack"])[c].shift(1)
    return d

def build_stacked_base_sample(df: pd.DataFrame) -> pd.DataFrame:
    cohort_years = list(range(COHORT_MIN, COHORT_MAX + 1))
    stacks = []
    for g in cohort_years:
        treated_ctys = df.loc[df["treat_year"] == g, "Country_Code"].dropna().unique().tolist()
        if not treated_ctys:
            continue

        # Controls: never-treated OR treated strictly after (g + W_POST)
        control_ctys = df.loc[
            df["treat_year"].isna() | (df["treat_year"] > (g + W_POST)),
            "Country_Code"
        ].dropna().unique().tolist()

        dfg = df[df["Country_Code"].isin(treated_ctys + control_ctys)].copy()
        dfg["stack"] = int(g)
        dfg["rel_year"] = dfg["Year"].astype(int) - int(g)
        dfg = dfg[(dfg["rel_year"] >= -W_PRE) & (dfg["rel_year"] <= W_POST)].copy()
        dfg["treated_stack"] = dfg["Country_Code"].isin(treated_ctys).astype(int)
        stacks.append(dfg)

    if not stacks:
        raise RuntimeError("No stacks created. Check COHORT_MIN/COHORT_MAX and treat_year coverage.")

    d = pd.concat(stacks, ignore_index=True)

    # Keep only (stack, rel_year) cells where both treated and control exist
    mm = d.groupby(["stack", "rel_year"])["treated_stack"].agg(["min", "max"])
    good = mm[(mm["min"] == 0) & (mm["max"] == 1)].index
    d = d.set_index(["stack", "rel_year"]).loc[good].reset_index()

    # time_id must be numeric/date-like for PanelOLS -> create numeric unique time_id per (stack, Year)
    d["time_id"] = d["Year"].astype(int) + 10000 * d["stack"].astype(int)  # numeric
    d["entity_id"] = d["Country_Code"].astype(str) + "__" + d["stack"].astype(str)

    return d

def make_event_terms(d: pd.DataFrame) -> tuple[pd.DataFrame, list, list]:
    k_list = [k for k in range(-W_PRE, W_POST + 1) if k != BASE_K]
    evt_terms = []
    for k in k_list:
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        d[term] = ((d["treated_stack"] == 1) & (d["rel_year"] == k)).astype(int)
        evt_terms.append(term)

    lead_terms = []
    for k in range(-W_PRE, 0):
        if k == BASE_K:
            continue
        if k <= -2:
            lead_terms.append(f"evt_{k:+d}".replace("+", "p").replace("-", "m"))
    return d, evt_terms, lead_terms

def wald_pretrend_p(res, lead_terms: list) -> float:
    if not lead_terms:
        return np.nan
    # Build R * beta = 0
    pindex = list(res.params.index)
    keep = [t for t in lead_terms if t in pindex]
    if not keep:
        return np.nan
    R = np.zeros((len(keep), len(pindex)))
    for i, t in enumerate(keep):
        R[i, pindex.index(t)] = 1.0
    w = res.wald_test(R, np.zeros(len(keep)))
    try:
        return float(w.pval)
    except Exception:
        # older linearmodels may use .pvalue
        return float(getattr(w, "pvalue", np.nan))

def run_eventstudy_panelols(d_base: pd.DataFrame, outcome: str, evt_terms: list, lead_terms: list, controls: list):
    dd = d_base.copy()
    if outcome not in dd.columns:
        raise RuntimeError(f"Outcome missing: {outcome}")

    dd[outcome] = pd.to_numeric(dd[outcome], errors="coerce")
    dd = dd.dropna(subset=[outcome]).copy()

    # add lag controls (within stack)
    dd = add_lag_controls(dd, controls)
    lag_cols = [f"lag1_{c}" for c in controls]

    # set panel index
    dd = dd.set_index(["entity_id", "time_id"]).sort_index()

    # y and X
    y = dd[outcome]
    X = dd[evt_terms + lag_cols].copy()

    # drop constant / no-variation columns (helps rank issues)
    good_cols = [c for c in X.columns if X[c].nunique(dropna=False) > 1]
    X = X[good_cols]

    # clusters by original country (not stack)
    clusters = dd.index.get_level_values(0).str.split("__").str[0]
    clusters = pd.Series(clusters.values, index=dd.index)

    mod = PanelOLS(
        y, X,
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True,
        check_rank=False,        # avoid "exog not full rank" hard-stop
    )
    res = mod.fit(cov_type="clustered", clusters=clusters)

    # table for event terms only (that survived absorption)
    rows = []
    for k in range(-W_PRE, W_POST + 1):
        if k == BASE_K:
            continue
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        if term in res.params.index:
            coef = float(res.params[term])
            se = float(res.std_errors[term])
            pval = float(res.pvalues[term])
        else:
            coef, se, pval = np.nan, np.nan, np.nan
        rows.append({"term": term, "k": k, "coef": coef, "se": se, "pval": pval})

    tab = pd.DataFrame(rows).sort_values("k")
    pre_p = wald_pretrend_p(res, lead_terms)

    return res, tab, pre_p

def plot_eventstudy(tab: pd.DataFrame, title: str, out_png: str):
    plt.figure(figsize=(11, 5))
    plt.axhline(0, linewidth=1)
    plt.axvline(BASE_K, linestyle="--", linewidth=1)
    plt.errorbar(tab["k"], tab["coef"], yerr=1.96 * tab["se"], fmt="o", capsize=3)
    plt.title(title)
    plt.xlabel("Event time k (years relative to quota year)")
    plt.ylabel("Effect vs baseline (k=-1)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


# -------------------------
# MAIN
# -------------------------
def main():
    ensure_dir(OUTDIR)
    ensure_dir(MERGED_DIR)

    # Load data
    df_main = read_main_panel(MAIN_PANEL_CSV)
    df_main = normalize_country_year(df_main)

    df_q = build_qarot_min_panel_if_needed()
    df_q = normalize_country_year(df_q)
    df_q["treat_year"] = pd.to_numeric(df_q["treat_year"], errors="coerce").astype("Int64")
    df_q["PARL"] = pd.to_numeric(df_q["PARL"], errors="coerce")

    # Merge: bring treat_year (and QAROT PARL as fallback)
    df = df_main.merge(df_q[["Country_Code", "Year", "treat_year", "PARL"]].rename(columns={"PARL": "PARL_qarot"}),
                       on=["Country_Code", "Year"], how="left")

    # Choose PARL outcome: main PARL if present else QAROT
    if "PARL" in df.columns:
        df["PARL"] = pd.to_numeric(df["PARL"], errors="coerce")
        df["PARL"] = df["PARL"].where(df["PARL"].notna(), df["PARL_qarot"])
    else:
        df["PARL"] = df["PARL_qarot"]

    df = df.drop(columns=["PARL_qarot"], errors="ignore")

    # Optional indices
    df = build_optional_indices(df)

    # Save merged
    df.to_csv(MERGED_CSV, index=False)
    print("Saved merged panel to:", MERGED_CSV)

    # Build stacked base sample once
    d_base = build_stacked_base_sample(df)

    print("STACKED BASE SAMPLE")
    print("n_obs:", len(d_base),
          "countries:", d_base["Country_Code"].nunique(),
          "treated countries (in stacks):", d_base.loc[d_base["treated_stack"] == 1, "Country_Code"].nunique())
    print("years:", int(d_base["Year"].min()), "-", int(d_base["Year"].max()))
    used = sorted(d_base["stack"].unique())
    print("cohorts used:", used[:5], "...", used[-5:])

    # Event dummies
    d_base, evt_terms, lead_terms = make_event_terms(d_base)

    # Outcomes to estimate
    outcomes = [
        "PARL",
        "Human_Capital",
        "Life_Expectancy",
        "Female_Labor_Force",
        "Female_Adult_Literacy",
        "Fertility",
        "Women_Business",
        "Women_Civil_Society",
        "edu_index_z",
        "health_index_z",
        "composite_hc_z",
    ]
    outcomes = [o for o in outcomes if o in d_base.columns]

    summaries = []
    for out in outcomes:
        try:
            res, tab, pre_p = run_eventstudy_panelols(d_base, out, evt_terms, lead_terms, CONTROL_COLS)

            # Save outputs
            tab_path = os.path.join(OUTDIR, f"eventstudy_point_{out}_w{W_PRE}_{W_POST}.csv")
            tab.to_csv(tab_path, index=False)

            png_path = os.path.join(OUTDIR, f"eventstudy_plot_{out}_w{W_PRE}_{W_POST}.png")
            avg_post_0_5 = float(tab.loc[(tab["k"] >= 0) & (tab["k"] <= 5), "coef"].mean())

            plot_eventstudy(
                tab,
                title=f"Stacked event-study: {out} (baseline k={BASE_K} omitted) | pretrend p={pre_p:.3f}",
                out_png=png_path
            )

            summaries.append({
                "outcome": out,
                "n_obs": int(res.nobs),
                "n_entities": int(res.entity_info.total),
                "pretrend_p": pre_p,
                "avg_post_0..5": avg_post_0_5,
                "csv": tab_path,
                "png": png_path,
            })

            print(f"[OK] {out}: pretrend_p={pre_p:.4f} avg_post_0..5={avg_post_0_5:.4f} -> {tab_path}")

        except Exception as e:
            print(f"[FAIL] {out}: {e}")

    if summaries:
        summ = pd.DataFrame(summaries).sort_values("outcome")
        summ_path = os.path.join(OUTDIR, "summary.csv")
        summ.to_csv(summ_path, index=False)
        print("\nSaved summary to:", summ_path)
    else:
        print("\nNo outcomes estimated (all failed or missing).")

    print("All outputs in:", OUTDIR)

main()


In [ ]:
# -*- coding: utf-8 -*-
"""
Stacked event-study for gender quota adoption (QAROT treat_year) on human-capital proxies.
Works in Jupyter and as a script.

Deps:
  pip install pandas numpy matplotlib linearmodels requests
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS

# ----------------------------
# CONFIG
# ----------------------------
MAIN_PANEL_CSV = "../data/final_data/final_data.csv"
QAROT_MIN_CSV  = "../data/qarot_data/qarot_panel_min.csv"
OUTDIR = "../data/eventstudy_outputs_panelols"

COHORT_MIN, COHORT_MAX = 2002, 2010
W_PRE, W_POST = 5, 12
BASE_K = -1

# Controls (будут использованы ТОЛЬКО как лаги)
CONTROL_COLS = ["GDP", "Political_Stability", "Corruption"]

# Optional WDI download
DOWNLOAD_WDI = True
WDI_INDICATORS = {
    "SE.SEC.ENRR.FE": "Female secondary enrollment, gross (%)",
    "SE.TER.ENRR.FE": "Female tertiary enrollment, gross (%)",
    "SE.PRM.ENRR.FE": "Female primary enrollment, gross (%)",
    "SE.ADT.LITR.FE.ZS": "Female adult literacy rate (%)",
    "SP.DYN.IMRT.IN": "Infant mortality rate (per 1,000 live births)",
    "SH.DYN.MORT": "Under-5 mortality rate (per 1,000 live births)",
    "SH.STA.MMRT": "Maternal mortality ratio (per 100,000 live births)",
}

BASE_OUTCOMES = [
    "PARL",
    "Life_Expectancy",
    "Female_Adult_Literacy",
    "Human_Capital",
    "Female_Labor_Force",
    "Fertility",
    "Women_Business",
    "Women_Civil_Society",
    "edu_index_z",
    "health_index_z",
    "composite_hc_z",
    # WDI raw (if downloaded)
    "SE.SEC.ENRR.FE",
    "SE.TER.ENRR.FE",
    "SE.PRM.ENRR.FE",
    "SE.ADT.LITR.FE.ZS",
    "SP.DYN.IMRT.IN",
    "SH.DYN.MORT",
    "SH.STA.MMRT",
]
# ----------------------------


def read_main_panel(path: str) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", engine="python", decimal=",")


def read_qarot_min(path: str) -> pd.DataFrame:
    return pd.read_csv(path)


def wb_fetch_indicator(ind_code: str, years: set) -> pd.DataFrame:
    import requests
    from urllib.parse import quote

    url = f"https://api.worldbank.org/v2/country/all/indicator/{quote(ind_code)}?format=json&per_page=20000"
    r = requests.get(url, timeout=90)
    r.raise_for_status()
    js = r.json()
    if not isinstance(js, list) or len(js) < 2:
        raise RuntimeError(f"Unexpected World Bank response for {ind_code}")
    data = js[1]

    rows = []
    for rec in data:
        iso3 = rec.get("countryiso3code")
        y = rec.get("date")
        v = rec.get("value")
        if not iso3 or not y:
            continue
        try:
            yy = int(y)
        except Exception:
            continue
        if yy not in years:
            continue
        rows.append((iso3, yy, v))

    df = pd.DataFrame(rows, columns=["Country_Code", "Year", ind_code])
    df[ind_code] = pd.to_numeric(df[ind_code], errors="coerce")
    return df


def zscore(s: pd.Series) -> pd.Series:
    m = s.mean(skipna=True)
    sd = s.std(skipna=True)
    if sd is None or sd == 0 or np.isnan(sd):
        return s * np.nan
    return (s - m) / sd


def build_indices(df: pd.DataFrame) -> pd.DataFrame:
    edu_candidates = []
    for c in ["Female_Adult_Literacy", "SE.ADT.LITR.FE.ZS", "SE.SEC.ENRR.FE", "SE.TER.ENRR.FE", "SE.PRM.ENRR.FE"]:
        if c in df.columns:
            edu_candidates.append(c)
    if edu_candidates:
        df["edu_index_z"] = pd.concat([zscore(df[c]) for c in edu_candidates], axis=1).mean(axis=1, skipna=True)
    else:
        df["edu_index_z"] = np.nan

    if "Life_Expectancy" in df.columns and "SP.DYN.IMRT.IN" in df.columns:
        df["health_index_z"] = zscore(df["Life_Expectancy"]) - zscore(df["SP.DYN.IMRT.IN"])
    elif "Life_Expectancy" in df.columns:
        df["health_index_z"] = zscore(df["Life_Expectancy"])
    else:
        df["health_index_z"] = np.nan

    parts = []
    for c in ["edu_index_z", "health_index_z"]:
        if c in df.columns:
            parts.append(df[c])
    if parts:
        df["composite_hc_z"] = pd.concat(parts, axis=1).mean(axis=1, skipna=True)
    else:
        df["composite_hc_z"] = np.nan

    return df


def add_lags(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.sort_values(["Country_Code", "Year"]).copy()
    for c in cols:
        if c in df.columns:
            df[f"lag1_{c}"] = df.groupby("Country_Code")[c].shift(1)
    return df


def build_stacked_sample(df: pd.DataFrame) -> pd.DataFrame:
    df = df[
        df["treat_year"].isna()
        | ((df["treat_year"] >= COHORT_MIN) & (df["treat_year"] <= COHORT_MAX))
    ].copy()

    cohort_years = list(range(COHORT_MIN, COHORT_MAX + 1))
    stacks = []

    for g in cohort_years:
        treated_ctys = df.loc[df["treat_year"] == g, "Country_Code"].dropna().unique().tolist()
        if not treated_ctys:
            continue

        control_ctys = df.loc[
            df["treat_year"].isna() | (df["treat_year"] > (g + W_POST)),
            "Country_Code"
        ].dropna().unique().tolist()

        dfg = df[df["Country_Code"].isin(treated_ctys + control_ctys)].copy()
        dfg["stack"] = int(g)
        dfg["rel_year"] = dfg["Year"].astype(int) - int(g)
        dfg = dfg[(dfg["rel_year"] >= -W_PRE) & (dfg["rel_year"] <= W_POST)].copy()
        dfg["treated_stack"] = dfg["Country_Code"].isin(treated_ctys).astype(int)
        stacks.append(dfg)

    if not stacks:
        raise RuntimeError("No stacks built. Check COHORT_MIN/COHORT_MAX.")

    d = pd.concat(stacks, ignore_index=True)

    mm = d.groupby(["stack", "rel_year"])["treated_stack"].agg(["min", "max"])
    good = mm[(mm["min"] == 0) & (mm["max"] == 1)].index
    d = d.set_index(["stack", "rel_year"]).loc[good].reset_index()

    d["entity_id"] = d["Country_Code"].astype(str) + "__" + d["stack"].astype(str)
    d["time_id"] = d["Year"].astype(int) * 10000 + d["stack"].astype(int)

    return d


def joint_pretrend_pvalue(res, lead_terms):
    """
    Joint Wald test H0: all lead_terms coefficients = 0.
    Compatible with older linearmodels (no `terms=`).
    """
    lead_terms = [t for t in lead_terms if t in res.params.index]
    if not lead_terms:
        return np.nan

    try:
        pindex = list(res.params.index)
        R = np.zeros((len(lead_terms), len(pindex)))
        for i, t in enumerate(lead_terms):
            R[i, pindex.index(t)] = 1.0
        q = np.zeros((len(lead_terms),))
        w = res.wald_test(R, q)
        if hasattr(w, "pval"):
            return float(w.pval)
        if hasattr(w, "pvalue"):
            return float(w.pvalue)
        return np.nan
    except Exception:
        return np.nan


def run_eventstudy_panelols(d: pd.DataFrame, outcome: str, controls: list):
    dd = d.copy()

    if outcome not in dd.columns:
        raise RuntimeError(f"Missing outcome: {outcome}")

    dd[outcome] = pd.to_numeric(dd[outcome], errors="coerce")
    dd = dd.dropna(subset=[outcome]).copy()

    k_list = [k for k in range(-W_PRE, W_POST + 1) if k != BASE_K]
    evt_terms = []
    for k in k_list:
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        dd[term] = ((dd["treated_stack"] == 1) & (dd["rel_year"] == k)).astype(float)
        evt_terms.append(term)

    use_controls = []
    for c in controls:
        lc = f"lag1_{c}"
        if lc in dd.columns:
            dd[lc] = pd.to_numeric(dd[lc], errors="coerce")
            use_controls.append(lc)

    dd = dd.set_index(["entity_id", "time_id"]).sort_index()

    y = dd[outcome].astype(float)
    X = dd[evt_terms + use_controls].astype(float)

    keep = [c for c in X.columns if X[c].nunique(dropna=False) > 1]
    X = X[keep]

    clusters = pd.Series(
        dd.index.get_level_values(0).str.split("__").str[0].values,
        index=dd.index
    )

    res = PanelOLS(
        y, X,
        entity_effects=True, time_effects=True,
        drop_absorbed=True, check_rank=False
    ).fit(cov_type="clustered", clusters=clusters)

    rows = []
    for k in k_list:
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        if term in res.params.index:
            rows.append({
                "outcome": outcome,
                "k": k,
                "term": term,
                "coef": float(res.params[term]),
                "se": float(res.std_errors[term]),
                "pval": float(res.pvalues[term]),
            })
        else:
            rows.append({"outcome": outcome, "k": k, "term": term, "coef": np.nan, "se": np.nan, "pval": np.nan})

    tab = pd.DataFrame(rows).sort_values("k")

    lead_terms = [f"evt_{k:+d}".replace("+", "p").replace("-", "m") for k in range(-W_PRE, 0) if k <= -2 and k != BASE_K]
    pre_p = joint_pretrend_pvalue(res, lead_terms)

    return res, tab, pre_p


def plot_eventstudy(tab: pd.DataFrame, title: str, out_png: str):
    plt.figure(figsize=(11, 5))
    plt.axhline(0, linewidth=1)
    plt.axvline(BASE_K, linestyle="--", linewidth=1)
    plt.errorbar(tab["k"], tab["coef"], yerr=1.96 * tab["se"], fmt="o", capsize=3)
    plt.title(title)
    plt.xlabel("Event time k (years relative to quota year)")
    plt.ylabel("Effect vs baseline (k=-1)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def main():
    os.makedirs(OUTDIR, exist_ok=True)

    df_main = read_main_panel(MAIN_PANEL_CSV)
    df_q = read_qarot_min(QAROT_MIN_CSV)

    if "year" in df_main.columns and "Year" not in df_main.columns:
        df_main = df_main.rename(columns={"year": "Year"})

    df_main["Year"] = pd.to_numeric(df_main["Year"], errors="coerce").astype("Int64")
    df_q["Year"] = pd.to_numeric(df_q["Year"], errors="coerce").astype("Int64")

    df = df_main.merge(df_q, on=["Country_Code", "Year"], how="left")

    if "treat_year" not in df.columns:
        raise RuntimeError("After merge, treat_year is missing. Check QAROT_MIN_CSV columns.")
    df["treat_year"] = pd.to_numeric(df["treat_year"], errors="coerce").astype("Int64")

    year_min = int(df["Year"].min())
    year_max = int(df["Year"].max())
    years = set(range(year_min, year_max + 1))

    if DOWNLOAD_WDI:
        wdi_merged = None
        for code in WDI_INDICATORS.keys():
            try:
                dfi = wb_fetch_indicator(code, years)
                wdi_merged = dfi if wdi_merged is None else wdi_merged.merge(dfi, on=["Country_Code", "Year"], how="outer")
                print(f"[WDI] {code} rows={len(dfi)}")
            except Exception as e:
                print(f"[WDI][WARN] {code} failed: {e}")
        if wdi_merged is not None:
            df = df.merge(wdi_merged, on=["Country_Code", "Year"], how="left")

    df = build_indices(df)
    df = add_lags(df, CONTROL_COLS)

    d = build_stacked_sample(df)

    print("STACKED BASE SAMPLE")
    print("n_obs:", len(d), "countries:", d["Country_Code"].nunique(), "treated countries (in stacks):", d.loc[d["treated_stack"]==1, "Country_Code"].nunique())
    print("years:", int(d["Year"].min()), "-", int(d["Year"].max()))
    print("cohorts used:", sorted(d["stack"].unique())[:5], "...", sorted(d["stack"].unique())[-5:])

    outcomes = [o for o in BASE_OUTCOMES if o in d.columns]

    summaries = []
    for out in outcomes:
        try:
            res, tab, pre_p = run_eventstudy_panelols(d, out, CONTROL_COLS)

            tab_path = os.path.join(OUTDIR, f"eventstudy_point_{out}_w{W_PRE}_{W_POST}.csv")
            tab.to_csv(tab_path, index=False)

            png_path = os.path.join(OUTDIR, f"eventstudy_plot_{out}_w{W_PRE}_{W_POST}.png")
            plot_eventstudy(tab, f"Stacked event-study: {out} (baseline k={BASE_K}) | pretrend p={pre_p}", png_path)

            avg_post_0_5 = tab[(tab["k"] >= 0) & (tab["k"] <= 5)]["coef"].mean()
            summaries.append({
                "outcome": out,
                "n_obs": int(res.nobs),
                "pretrend_p": pre_p,
                "avg_post_0..5": float(avg_post_0_5),
                "csv": tab_path,
                "png": png_path,
            })

            print(f"[OK] {out}: pretrend_p={pre_p} avg_post_0..5={avg_post_0_5:.4f} -> {tab_path}")

        except Exception as e:
            print(f"[FAIL] {out}: {e}")

    if summaries:
        summ = pd.DataFrame(summaries).sort_values("outcome")
        summ_path = os.path.join(OUTDIR, "summary.csv")
        summ.to_csv(summ_path, index=False)
        print("\nSaved summary to:", summ_path)
    else:
        print("\nNo outcomes estimated (all failed or missing).")

    print("All outputs in:", OUTDIR)


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from linearmodels.panel import PanelOLS


# =========================
# CONFIG
# =========================
MAIN_PANEL_CSV = "../data/final_data/final_data.csv"
QAROT_MIN_CSV  = "../data/qarot_data/qarot_panel_min.csv"

MERGED_DIR     = "../data/merged_panel"
MERGED_CSV     = os.path.join(MERGED_DIR, "merged_panel.csv")

OUTDIR         = "../data/eventstudy_outputs_panelols"

COHORT_MIN, COHORT_MAX = 2002, 2010
W_PRE, W_POST          = 5, 12
BASE_K                 = -1

# Контролы по умолчанию выключены, чтобы скрипт был максимально “железный”.
# Если захочешь — добавь сюда колонки из final_data (будут использованы как lag1_*)
CONTROL_COLS = []

OUTCOMES = [
    "PARL",
    "Human_Capital",
    "Life_Expectancy",
    "Female_Labor_Force",
    "Female_Adult_Literacy",
    "Fertility",
    "Women_Business",
    "Women_Civil_Society",
    "edu_index_z",
    "health_index_z",
    "composite_hc_z",
]
# =========================


def read_csv_smart(path: str) -> pd.DataFrame:
    """
    У тебя final_data.csv оказался с sep=';' и decimal=','.
    Этот загрузчик пробует несколько типичных вариантов.
    """
    tries = [
        dict(sep=";", engine="python", decimal=","),
        dict(sep=";", engine="python", decimal="."),
        dict(sep=",", engine="python", decimal="."),
        dict(sep=",", engine="python", decimal=","),
    ]
    last_err = None
    for kw in tries:
        try:
            df = pd.read_csv(path, **kw)
            return df
        except Exception as e:
            last_err = e
    raise last_err


def zscore(s: pd.Series) -> pd.Series:
    m = s.mean(skipna=True)
    sd = s.std(skipna=True)
    if sd is None or sd == 0 or np.isnan(sd):
        return pd.Series(np.nan, index=s.index)
    return (s - m) / sd


def build_optional_indices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Индексы строятся из того, что реально есть в merged панеле.
    - edu_index_z: среднее z-score по доступным образоват. компонентам
    - health_index_z: z(Life_Expectancy) - z(mortality/imr) (если есть)
    - composite_hc_z: среднее из доступных {edu_index_z, health_index_z, z(Human_Capital), z(pwt_hc)}
    """
    cols = set(df.columns)

    # Education components (берём всё, что есть)
    edu_candidates = [
        "SE.SEC.ENRR.FE", "SE.TER.ENRR.FE", "SE.PRM.ENRR.FE",
        "SE.ADT.LITR.FE.ZS",
        "Female_Adult_Literacy",
    ]
    edu_components = [c for c in edu_candidates if c in cols]

    if edu_components:
        df["edu_index_z"] = pd.concat([zscore(df[c]) for c in edu_components], axis=1).mean(axis=1, skipna=True)
    else:
        df["edu_index_z"] = np.nan

    # Health: life expectancy + (optional) mortality
    mort_candidates = ["SP.DYN.IMRT.IN", "SH.DYN.MORT", "SH.STA.MMRT"]
    mort_col = next((c for c in mort_candidates if c in cols), None)

    if "Life_Expectancy" in cols and mort_col is not None:
        df["health_index_z"] = zscore(df["Life_Expectancy"]) - zscore(df[mort_col])
    elif "Life_Expectancy" in cols:
        df["health_index_z"] = zscore(df["Life_Expectancy"])
    elif mort_col is not None:
        df["health_index_z"] = -zscore(df[mort_col])
    else:
        df["health_index_z"] = np.nan

    # Composite
    comp_parts = []
    for c in ["edu_index_z", "health_index_z", "Human_Capital", "pwt_hc"]:
        if c in cols:
            comp_parts.append(zscore(df[c]) if c in ["Human_Capital", "pwt_hc"] else df[c])
    df["composite_hc_z"] = pd.concat(comp_parts, axis=1).mean(axis=1, skipna=True) if comp_parts else np.nan

    return df


def normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    # Country_Code
    if "Country_Code" not in df.columns:
        raise RuntimeError("Main panel must contain Country_Code.")
    if "Year" not in df.columns:
        raise RuntimeError("Main panel must contain Year.")

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df


def add_lag_controls(df: pd.DataFrame, controls: list[str]) -> pd.DataFrame:
    if not controls:
        return df
    for c in controls:
        if c not in df.columns:
            df[c] = np.nan
    df = df.sort_values(["Country_Code", "Year"])
    for c in controls:
        df[f"lag1_{c}"] = df.groupby("Country_Code")[c].shift(1)
    return df


def load_and_merge() -> pd.DataFrame:
    os.makedirs(MERGED_DIR, exist_ok=True)

    df_main = read_csv_smart(MAIN_PANEL_CSV)
    df_main = normalize_keys(df_main)

    df_q = pd.read_csv(QAROT_MIN_CSV)
    # expected: Country_Code, Year, treat_year, PARL
    for c in ["Country_Code", "Year", "treat_year", "PARL"]:
        if c not in df_q.columns:
            raise RuntimeError(f"QAROT minimal panel must contain {c}.")
    df_q["Year"] = pd.to_numeric(df_q["Year"], errors="coerce").astype("Int64")
    df_q["treat_year"] = pd.to_numeric(df_q["treat_year"], errors="coerce").astype("Int64")

    df = df_main.merge(df_q, on=["Country_Code", "Year"], how="left")
    df.to_csv(MERGED_CSV, index=False)
    return df


def build_stacked_sample(df: pd.DataFrame) -> pd.DataFrame:
    # restrict cohorts: treated in [COHORT_MIN, COHORT_MAX] + never-treated
    df = df[(df["treat_year"].isna()) | ((df["treat_year"] >= COHORT_MIN) & (df["treat_year"] <= COHORT_MAX))].copy()

    cohort_years = list(range(COHORT_MIN, COHORT_MAX + 1))
    stacks = []

    for g in cohort_years:
        treated_ctys = df.loc[df["treat_year"] == g, "Country_Code"].dropna().unique().tolist()
        if not treated_ctys:
            continue

        # controls: never-treated OR treated later than g + W_POST
        control_ctys = df.loc[df["treat_year"].isna() | (df["treat_year"] > (g + W_POST)), "Country_Code"] \
                        .dropna().unique().tolist()

        dfg = df[df["Country_Code"].isin(treated_ctys + control_ctys)].copy()
        dfg["stack"] = int(g)
        dfg["rel_year"] = dfg["Year"].astype(int) - int(g)
        dfg = dfg[(dfg["rel_year"] >= -W_PRE) & (dfg["rel_year"] <= W_POST)].copy()
        dfg["treated_stack"] = dfg["Country_Code"].isin(treated_ctys).astype(int)

        stacks.append(dfg)

    if not stacks:
        raise RuntimeError("No stacks created. Check COHORT_MIN/COHORT_MAX.")

    d = pd.concat(stacks, ignore_index=True)

    # Keep only (stack, rel_year) cells where both treated and control exist
    mm = d.groupby(["stack", "rel_year"])["treated_stack"].agg(["min", "max"])
    good_idx = mm[(mm["min"] == 0) & (mm["max"] == 1)].index
    d = d.set_index(["stack", "rel_year"]).loc[good_idx].reset_index()

    # FE ids: entity = country×stack ; time = year×stack (numeric id)
    d["fe_unit"] = d["Country_Code"].astype(str) + "__" + d["stack"].astype(str)
    d["fe_year_id"] = d["stack"].astype(int) * 10000 + d["Year"].astype(int)  # numeric, unique per stack-year

    return d


def wald_pretrend(res, lead_terms: list[str]) -> float:
    """
    Joint test that all lead coefficients == 0.
    Works with linearmodels results by passing an R matrix.
    Returns p-value (float) or nan.
    """
    if not lead_terms:
        return float("nan")

    params = list(res.params.index)
    keep = [t for t in lead_terms if t in params]
    if not keep:
        return float("nan")

    R = np.zeros((len(keep), len(params)))
    for i, t in enumerate(keep):
        R[i, params.index(t)] = 1.0

    try:
        w = res.wald_test(R)
        return float(w.pval)
    except Exception:
        return float("nan")


def run_eventstudy_panelols(d: pd.DataFrame, outcome: str, controls: list[str]):
    dd = d.copy()

    # Event dummies
    k_list = [k for k in range(-W_PRE, W_POST + 1) if k != BASE_K]
    evt_terms = []
    for k in k_list:
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        dd[term] = ((dd["treated_stack"] == 1) & (dd["rel_year"] == k)).astype(int)
        evt_terms.append(term)

    lead_terms = [
        f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        for k in range(-W_PRE, 0) if (k != BASE_K and k <= -2)
    ]

    # Build panel index
    dd = dd.set_index(["fe_unit", "fe_year_id"]).sort_index()

    # y, X
    if outcome not in dd.columns:
        raise RuntimeError(f"Outcome not found: {outcome}")

    xcols = evt_terms + [f"lag1_{c}" for c in controls if f"lag1_{c}" in dd.columns]
    use_cols = [outcome] + xcols
    dd2 = dd[use_cols].dropna()

    y = dd2[outcome]
    X = dd2[xcols]

    # Drop non-varying columns (avoids rank errors)
    nunq = X.nunique(dropna=False)
    X = X.loc[:, nunq > 1]

    # Cluster by base country (fe_unit = ISO3__stack -> ISO3)
    clusters = pd.Series(
        dd2.index.get_level_values(0).str.split("__").str[0].values,
        index=dd2.index
    )

    res = PanelOLS(
        y, X,
        entity_effects=True,
        time_effects=True,
        check_rank=False,
        drop_absorbed=True
    ).fit(cov_type="clustered", clusters=clusters)

    # Coef table (only event terms)
    rows = []
    for k in range(-W_PRE, W_POST + 1):
        if k == BASE_K:
            continue
        term = f"evt_{k:+d}".replace("+", "p").replace("-", "m")
        rows.append({
            "k": k,
            "term": term,
            "coef": float(res.params.get(term, np.nan)),
            "se": float(res.std_errors.get(term, np.nan)),
            "pval": float(res.pvalues.get(term, np.nan)),
        })
    tab = pd.DataFrame(rows).sort_values("k")

    pre_p = wald_pretrend(res, lead_terms)

    return res, tab, pre_p


def plot_eventstudy(tab: pd.DataFrame, title: str, out_png: str, base_k: int):
    plt.figure(figsize=(12, 6))
    plt.axhline(0, linewidth=1)
    plt.axvline(base_k, linestyle="--", linewidth=1)

    yerr = 1.96 * tab["se"].values
    plt.errorbar(tab["k"], tab["coef"], yerr=yerr, fmt="o", capsize=3)

    plt.title(title)
    plt.xlabel("Event time k")
    plt.ylabel(f"Effect vs baseline (k={base_k})")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def main():
    os.makedirs(OUTDIR, exist_ok=True)

    df = load_and_merge()
    df = build_optional_indices(df)
    df = add_lag_controls(df, CONTROL_COLS)

    d = build_stacked_sample(df)

    print("STACKED BASE SAMPLE")
    print("n_obs:", len(d),
          "countries:", d["Country_Code"].nunique(),
          "treated countries (in stacks):", d.loc[d["treated_stack"] == 1, "Country_Code"].nunique())
    print("years:", int(d["Year"].min()), "-", int(d["Year"].max()))
    print("cohorts used:", sorted(d["stack"].unique())[:5], "...", sorted(d["stack"].unique())[-5:])

    summaries = []

    for out in OUTCOMES:
        if out not in d.columns:
            continue

        # avoid "control == outcome"
        controls = [c for c in CONTROL_COLS if c != out]

        try:
            res, tab, pre_p = run_eventstudy_panelols(d, out, controls)

            tab_path = os.path.join(OUTDIR, f"eventstudy_point_{out}_w{W_PRE}_{W_POST}.csv")
            tab.to_csv(tab_path, index=False)

            avg_post_0_5 = float(tab[(tab["k"] >= 0) & (tab["k"] <= 5)]["coef"].mean())

            png_path = os.path.join(OUTDIR, f"eventstudy_plot_{out}_w{W_PRE}_{W_POST}.png")
            title = f"Stacked ES (PanelOLS): {out} | cohorts {COHORT_MIN}-{COHORT_MAX} | window [-{W_PRE},+{W_POST}] | pretrend p={pre_p:.3f}"
            plot_eventstudy(tab, title, png_path, BASE_K)

            summaries.append({
                "outcome": out,
                "n_obs": int(res.nobs),
                "avg_post_0_5": avg_post_0_5,
                "pretrend_p": pre_p,
                "csv": tab_path,
                "png": png_path,
            })

            print(f"[OK] {out}: pretrend_p={pre_p:.4f} avg_post_0..5={avg_post_0_5:.4f} -> {tab_path}")

        except Exception as e:
            print(f"[FAIL] {out}: {e}")

    if summaries:
        summ = pd.DataFrame(summaries).sort_values("outcome")
        summ_path = os.path.join(OUTDIR, "summary.csv")
        summ.to_csv(summ_path, index=False)
        print("\nSaved summary to:", summ_path)
    else:
        print("\nNo outcomes estimated (all failed or missing).")

    print("Merged panel saved to:", MERGED_CSV)
    print("All outputs in:", OUTDIR)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# =========================================================
# НАСТРОЙКИ
# =========================================================
DATA_PATH = "../data/final_data/final_data.csv"
OUT_DIR = Path("../data/parl_descriptive_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN = 1997
YEAR_MAX = 2015
MIN_COUNTRIES_IN_REGION = 3

# Белый фон по умолчанию для всех графиков
pio.templates.default = "plotly_white"

# Палитра с большим числом разных цветов
COLOR_SEQUENCE = px.colors.qualitative.Dark24

# Сокращения длинных названий регионов
REGION_NAME_MAP = {
    "Central America and the Caribbean": "Central Am. & Caribbean",
    "Sub-Saharan Africa": "Sub-Saharan Africa",
    "Western Europe": "Western Europe",
    "Central Europe": "Central Europe",
    "Eastern Europe": "Eastern Europe",
    "South America": "South America",
    "Central Asia": "Central Asia",
    "East Asia": "East Asia",
    "South Asia": "South Asia",
    "North Africa": "North Africa",
    "Middle East": "Middle East",
    "Oceania": "Oceania",
}

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================================================
def save_plot(fig, file_name: str, width=1400, height=900):
    """
    Сохраняет график в HTML и PNG.
    Для PNG нужен kaleido: pip install -U kaleido
    """
    html_path = OUT_DIR / f"{file_name}.html"
    png_path = OUT_DIR / f"{file_name}.png"

    fig.write_html(str(html_path))

    try:
        fig.write_image(str(png_path), width=width, height=height, scale=2)
        print(f"PNG сохранён: {png_path}")
    except Exception as e:
        print(f"Не удалось сохранить PNG для {file_name}")
        print("Проверь, установлен ли kaleido: pip install -U kaleido")
        print("HTML всё равно сохранён.")
        print("Ошибка:", e)

    print(f"HTML сохранён: {html_path}")


def apply_common_layout(fig, legend_title="Регион", bottom_margin=170):
    """
    Общие настройки оформления.
    """
    fig.update_layout(
        width=1400,
        height=900,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=80, r=40, t=80, b=bottom_margin),
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.22,
            xanchor="center",
            x=0.5,
            title_text=legend_title
        )
    )
    return fig


# =========================================================
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# =========================================================
df = pd.read_csv(DATA_PATH, sep=";", decimal=",")

# Приведение типов
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["PARL"] = pd.to_numeric(df["PARL"], errors="coerce")

# Рабочий период
df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()

# Убираем строки без ключевых полей
df = df.dropna(subset=["PARL", "Region", "Country_Name"]).copy()

# Оставляем только регионы с достаточным числом стран
region_counts = df.groupby("Region")["Country_Name"].nunique().sort_values(ascending=False)
keep_regions = region_counts[region_counts >= MIN_COUNTRIES_IN_REGION].index.tolist()
df = df[df["Region"].isin(keep_regions)].copy()

# Сокращённые названия регионов
df["Region_short"] = df["Region"].replace(REGION_NAME_MAP)

# =========================================================
# ГРАФИК 1. ДИНАМИКА СРЕДНЕГО PARL ПО РЕГИОНАМ
# =========================================================
region_year = (
    df.groupby(["Year", "Region_short"], as_index=False)["PARL"]
      .mean()
      .sort_values(["Region_short", "Year"])
)

# Порядок регионов в легенде: от более высокого среднего PARL к более низкому
region_order = (
    region_year.groupby("Region_short")["PARL"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig1 = px.line(
    region_year,
    x="Year",
    y="PARL",
    color="Region_short",
    markers=True,
    category_orders={"Region_short": region_order},
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Средняя доля женщин в парламенте по регионам, 1997–2015",
    labels={
        "Year": "Год",
        "PARL": "PARL, %",
        "Region_short": "Регион"
    }
)

fig1.update_traces(line=dict(width=2), marker=dict(size=6))
fig1 = apply_common_layout(fig1, legend_title="Регион", bottom_margin=190)

save_plot(fig1, "parl_region_trends_1997_2015")


# =========================================================
# ГРАФИК 2. ТОП-15 СТРАН ПО СРЕДНЕМУ PARL
# =========================================================
country_mean = (
    df.groupby("Country_Name", as_index=False)["PARL"]
      .mean()
      .sort_values("PARL", ascending=False)
      .head(15)
      .sort_values("PARL", ascending=True)
)

fig2 = px.bar(
    country_mean,
    x="PARL",
    y="Country_Name",
    orientation="h",
    title="Страны с самым высоким средним уровнем PARL, 1997–2015",
    labels={
        "PARL": "Средняя доля женщин в парламенте, %",
        "Country_Name": "Страна"
    },
    text="PARL"
)

fig2.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig2.update_layout(
    width=1400,
    height=900,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=80, r=80, t=80, b=80)
)

save_plot(fig2, "parl_top15_countries_mean_1997_2015")


# =========================================================
# ГРАФИК 3. РАСПРЕДЕЛЕНИЕ СРЕДНЕГО PARL ПО СТРАНАМ ВНУТРИ РЕГИОНОВ
# =========================================================
country_region_mean = (
    df.groupby(["Region_short", "Country_Name"], as_index=False)["PARL"]
      .mean()
)

region_box_order = (
    country_region_mean.groupby("Region_short")["PARL"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig3 = px.box(
    country_region_mean,
    x="Region_short",
    y="PARL",
    category_orders={"Region_short": region_box_order},
    points="all",
    title="Распределение среднего PARL по странам внутри регионов, 1997–2015",
    labels={
        "Region_short": "Регион",
        "PARL": "Средняя доля женщин в парламенте, %"
    }
)

fig3.update_layout(
    width=1500,
    height=900,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=80, r=40, t=80, b=150)
)
fig3.update_xaxes(tickangle=35)

save_plot(fig3, "parl_region_boxplot_country_means", width=1500, height=900)


# =========================================================
# СВОДНАЯ ТАБЛИЦА ПО РЕГИОНАМ
# =========================================================
region_summary = (
    df.groupby("Region_short", as_index=False)
      .agg(
          mean_parl=("PARL", "mean"),
          median_parl=("PARL", "median"),
          countries=("Country_Name", "nunique"),
          observations=("PARL", "count")
      )
      .sort_values("mean_parl", ascending=False)
)

region_summary.to_csv(OUT_DIR / "parl_region_summary.csv", index=False)
print("CSV сохранён:", OUT_DIR / "parl_region_summary.csv")

print("\nГотово. Файлы лежат в папке:")
print(OUT_DIR)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# =========================================================
# НАСТРОЙКИ
# =========================================================
DATA_PATH = "../data/final_data/final_data.csv"
OUT_DIR = Path("../data/hc_descriptive_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN = 1997
YEAR_MAX = 2015
MIN_COUNTRIES_IN_REGION = 3

# Белый фон по умолчанию
pio.templates.default = "plotly_white"

# Палитра с достаточным числом разных цветов
COLOR_SEQUENCE = px.colors.qualitative.Dark24

# Сокращения названий регионов
REGION_NAME_MAP = {
    "Central America and the Caribbean": "Central Am. & Caribbean",
    "Sub-Saharan Africa": "Sub-Saharan Africa",
    "Western Europe": "Western Europe",
    "Central Europe": "Central Europe",
    "Eastern Europe": "Eastern Europe",
    "South America": "South America",
    "Central Asia": "Central Asia",
    "East Asia": "East Asia",
    "South Asia": "South Asia",
    "North Africa": "North Africa",
    "Middle East": "Middle East",
    "Oceania": "Oceania",
}

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================================================
def save_plot(fig, file_name: str, width=1400, height=900):
    """
    Сохраняет график в html и png.
    Для png нужен kaleido:
        pip install -U kaleido
    """
    html_path = OUT_DIR / f"{file_name}.html"
    png_path = OUT_DIR / f"{file_name}.png"

    fig.write_html(str(html_path))

    try:
        fig.write_image(str(png_path), width=width, height=height, scale=2)
        print(f"PNG сохранён: {png_path}")
    except Exception as e:
        print(f"Не удалось сохранить PNG для {file_name}")
        print("Проверь, установлен ли kaleido: pip install -U kaleido")
        print("HTML всё равно сохранён.")
        print("Ошибка:", e)

    print(f"HTML сохранён: {html_path}")


def apply_line_layout(fig, legend_title="Регион", bottom_margin=190):
    fig.update_layout(
        width=1400,
        height=900,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=80, r=40, t=80, b=bottom_margin),
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.22,
            xanchor="center",
            x=0.5,
            title_text=legend_title
        )
    )
    return fig


# =========================================================
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# =========================================================
df = pd.read_csv(DATA_PATH, sep=";", decimal=",")

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Human_Capital"] = pd.to_numeric(df["Human_Capital"], errors="coerce")

df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()
df = df.dropna(subset=["Human_Capital", "Region", "Country_Name"]).copy()

region_counts = df.groupby("Region")["Country_Name"].nunique().sort_values(ascending=False)
keep_regions = region_counts[region_counts >= MIN_COUNTRIES_IN_REGION].index.tolist()
df = df[df["Region"].isin(keep_regions)].copy()

df["Region_short"] = df["Region"].replace(REGION_NAME_MAP)

# =========================================================
# ГРАФИК 1. СРЕДНИЙ HUMAN CAPITAL ПО РЕГИОНАМ ВО ВРЕМЕНИ
# =========================================================
region_year = (
    df.groupby(["Year", "Region_short"], as_index=False)["Human_Capital"]
      .mean()
      .sort_values(["Region_short", "Year"])
)

region_order = (
    region_year.groupby("Region_short")["Human_Capital"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig1 = px.line(
    region_year,
    x="Year",
    y="Human_Capital",
    color="Region_short",
    markers=True,
    category_orders={"Region_short": region_order},
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Средний индекс человеческого капитала по регионам, 1997–2015",
    labels={
        "Year": "Год",
        "Human_Capital": "Индекс человеческого капитала",
        "Region_short": "Регион"
    }
)

fig1.update_traces(line=dict(width=2), marker=dict(size=6))
fig1 = apply_line_layout(fig1, legend_title="Регион", bottom_margin=190)

save_plot(fig1, "hc_region_trends_1997_2015")


# =========================================================
# ГРАФИК 2. ТОП-15 СТРАН ПО СРЕДНЕМУ HUMAN CAPITAL
# =========================================================
country_mean = (
    df.groupby("Country_Name", as_index=False)["Human_Capital"]
      .mean()
      .sort_values("Human_Capital", ascending=False)
      .head(15)
      .sort_values("Human_Capital", ascending=True)
)

fig2 = px.bar(
    country_mean,
    x="Human_Capital",
    y="Country_Name",
    orientation="h",
    title="Страны с самым высоким средним уровнем человеческого капитала, 1997–2015",
    labels={
        "Human_Capital": "Средний индекс человеческого капитала",
        "Country_Name": "Страна"
    },
    text="Human_Capital"
)

fig2.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig2.update_layout(
    width=1400,
    height=900,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=80, r=80, t=80, b=80)
)

save_plot(fig2, "hc_top15_countries_mean_1997_2015")


# =========================================================
# ГРАФИК 3. РАСПРЕДЕЛЕНИЕ СРЕДНЕГО HUMAN CAPITAL ПО СТРАНАМ ВНУТРИ РЕГИОНОВ
# =========================================================
country_region_mean = (
    df.groupby(["Region_short", "Country_Name"], as_index=False)["Human_Capital"]
      .mean()
)

region_box_order = (
    country_region_mean.groupby("Region_short")["Human_Capital"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig3 = px.box(
    country_region_mean,
    x="Region_short",
    y="Human_Capital",
    category_orders={"Region_short": region_box_order},
    points="all",
    title="Распределение среднего индекса человеческого капитала по странам внутри регионов, 1997–2015",
    labels={
        "Region_short": "Регион",
        "Human_Capital": "Средний индекс человеческого капитала"
    }
)

fig3.update_layout(
    width=1500,
    height=900,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=80, r=40, t=80, b=150)
)
fig3.update_xaxes(tickangle=35)

save_plot(fig3, "hc_region_boxplot_country_means", width=1500, height=900)


# =========================================================
# СВОДНАЯ ТАБЛИЦА ПО РЕГИОНАМ
# =========================================================
region_summary = (
    df.groupby("Region_short", as_index=False)
      .agg(
          mean_hc=("Human_Capital", "mean"),
          median_hc=("Human_Capital", "median"),
          countries=("Country_Name", "nunique"),
          observations=("Human_Capital", "count")
      )
      .sort_values("mean_hc", ascending=False)
)

region_summary.to_csv(OUT_DIR / "hc_region_summary.csv", index=False)
print("CSV сохранён:", OUT_DIR / "hc_region_summary.csv")

print("\nГотово. Все файлы лежат в папке:")
print(OUT_DIR)